# Mission 2 — Stage 12 Raw-waveform Pretrained Speech

Frozen WavLM Base+ representation probe. This Notebook is the sole Stage 12 execution path; it contains CP0–CP10 in order, keeps Official Validation out of every stage, and halts at CP5 when the 20-hour compute gate fails. It never starts Stage 13.

Before running: copy this Notebook and `src/dcc_data/mission2_raw_waveform_pretrained_speech.py` into the existing Colab Drive repo at `/content/drive/MyDrive/prelim`. Confirm no Stage 11 cell is currently executing; if one is, cancel only that Stage 11 cell and retain the runtime. The Notebook does not terminate processes or reset the runtime.

The source data path must already exist in the current Colab runtime. CP1 reuses the saved Stage 11 transfer proof, manifest hashes and fixed-split identity, checks the existing raw folder path and file count, and refuses to download, extract, rebuild, or silently substitute data. It does not reread all WAV bytes for another 16-minute content hash.


## CP0 — Environment and dependencies

Use the current Colab GPU/runtime. Torch and torchaudio are not upgraded or replaced. If Transformers is absent, CP0 installs the pinned Transformers package only; any import/CUDA incompatibility stops the Notebook. The active device and exact package versions are saved for this run.


In [2]:
from pathlib import Path
import importlib
import importlib.metadata
import importlib.util
import subprocess
import sys
import time

from google.colab import drive
drive.mount('/content/drive')
REPO_ROOT = Path('/content/drive/MyDrive/prelim').resolve()
HELPER_PATH = REPO_ROOT / 'src/dcc_data/mission2_raw_waveform_pretrained_speech.py'
if not HELPER_PATH.is_file():
    raise FileNotFoundError(f'Copy the Stage 12 helper into the Drive repo first: {HELPER_PATH}')
sys.path.insert(0, str(REPO_ROOT / 'src'))

def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

# Keep Colab's Torch/CUDA stack unchanged. Install only the pinned HF client if absent.
if importlib.util.find_spec('transformers') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers==4.57.1'], check=True)
import torch
import torchaudio
import pandas
import pyarrow
import sklearn
import joblib
if not torch.cuda.is_available():
    raise RuntimeError('Stage 12 compute gate requires the current Colab CUDA GPU; CPU fallback is not allowed.')
if package_version('torch').split('+')[0].rsplit('.', 1)[0] != package_version('torchaudio').split('+')[0].rsplit('.', 1)[0]:
    raise RuntimeError('Torch/torchaudio version mismatch; do not replace the Colab Torch stack automatically.')
from transformers import AutoFeatureExtractor, WavLMModel
from dcc_data import mission2_raw_waveform_pretrained_speech as stage12
stage12 = importlib.reload(stage12)

RAW_ROOT = Path('/content/mission2_shared_training_raw').resolve()
SHARED_ARCHIVE_ROOT = REPO_ROOT / 'data/raw_original/대학부 데이터'
TRANSFER_IDENTITY_PATH = REPO_ROOT / 'tmp/mission2_egemaps_transfer_identity.json'
STAGE11_IDENTITY_PATH = REPO_ROOT / stage12.STAGE11_RUN / 'data_identity.json'
REFERENCE_ROOT = REPO_ROOT / stage12.REFERENCE_RUN
OUTPUT_PARENT = REPO_ROOT / 'data/processed/mission2_raw_waveform_pretrained_speech'
ENVIRONMENT = stage12.environment_info()
print({'environment': ENVIRONMENT, 'raw_root_exists': RAW_ROOT.is_dir(),
       'transfer_proof_exists': TRANSFER_IDENTITY_PATH.is_file(),
       'stage11_identity_exists': STAGE11_IDENTITY_PATH.is_file(),
       'reference_root_exists': REFERENCE_ROOT.is_dir(),
       'shared_archive_root_exists': SHARED_ARCHIVE_ROOT.is_dir()})


Mounted at /content/drive
{'environment': {'python': '3.13.15', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39', 'packages': {'numpy': '2.1.3', 'pandas': '2.2.3', 'pyarrow': '23.0.1', 'scikit-learn': '1.6.1', 'torch': '2.11.0+cu130', 'torchaudio': '2.11.0+cu130', 'transformers': '5.18.0', 'safetensors': '0.8.0', 'opensmile': None}, 'device': 'cuda', 'gpu': 'NVIDIA RTX PRO 6000 Blackwell Server Edition', 'cuda': '13.0', 'implementation_sha256': '9edf144bb8a9a6ed7d66ad220aca51f70a416a842211c6f19e0741895999c081', 'torch_import_error': None}, 'raw_root_exists': False, 'transfer_proof_exists': True, 'stage11_identity_exists': True, 'reference_root_exists': True, 'shared_archive_root_exists': True}


In [4]:
import json, subprocess, sys
report_path='/tmp/mission2_opensmile_pip_plan.json'
proc=subprocess.run([sys.executable,'-m','pip','install','--dry-run','--report',report_path,'opensmile==2.6.0'],capture_output=True,text=True)
print('returncode:',proc.returncode)
print(proc.stdout)
print(proc.stderr)
if proc.returncode==0:
    plan=json.load(open(report_path))
    print('PLANNED INSTALLS:',[(item['metadata']['name'],item['metadata']['version']) for item in plan.get('install',[])])
subprocess, sys
proc = subprocess.run([sys.executable, '-m', 'pip', 'install', 'opensmile==2.6.0'], capture_output=True, text=True)
print(proc.stdout)
print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError(f'opensmile install failed: {proc.returncode}')
import opensmile
from importlib.metadata import version
assert version('opensmile') == '2.6.0'
from importlib.metadata import PackageNotFoundError, version
import subprocess, sys
try:
    installed = version('opensmile')
except PackageNotFoundError:
    installed = None
if installed != '2.6.0':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'opensmile==2.6.0'], check=True)
import opensmile
assert version('opensmile') == '2.6.0'
print({'status': 'PASS', 'opensmile': version('opensmile')})

returncode: 0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.0/325.0 kB 40.2 MB/s eta 0:00:00
Would install asttokens-3.0.2 audeer-2.6.0 audformat-1.4.2 audinterface-1.4.0 audiofile-1.6.1 audmath-1.4.4 audobject-0.7.13 audresample-1.3.6 iso3166-3.0.0 iso639-lang-2.6.3 opensmile-2.6.0 oyaml-1.0


PLANNED INSTALLS: [('opensmile', '2.6.0'), ('audinterface', '1.4.0'), ('audobject', '0.7.13'), ('asttokens', '3.0.2'), ('audeer', '2.6.0'), ('audformat', '1.4.2'), ('audiofile', '1.6.1'), ('audmath', '1.4.4'), ('audresample', '1.3.6'), ('oyaml', '1.0'), ('iso3166', '3.0.0'), ('iso639-lang', '2.6.3')]
  Using cached opensmile-2.6.0-py3-none-manylinux_2_17_x86_64.whl.metadata (15 kB)
  Using cached audobject-0.7.13-py3-none-any.whl.metadata (2.6 kB)
  Using cached

## CP1 — Organizer provenance, existing rows, and fixed call split

Read the already cached Training manifest and saved Stage 11 proof only. Require the clean 59D fingerprint, exact saved manifest/proof hashes, row/key/call/label/crop identity, fixed `call_id` split, 711,467/177,891 counts, and zero call overlap. A missing file or mismatch stops here. No model is loaded and no WAV is re-extracted. If the prior ephemeral `/content` WAV tree is gone, CP1 indexes the same existing `data/raw_original/대학부 데이터` ZIP members and streams WAVs on demand. The saved full byte-level proof is reused; no full raw-content rehash occurs.


In [5]:
context, DATA_PREFLIGHT = stage12.load_existing_training_context(
    REPO_ROOT, RAW_ROOT, TRANSFER_IDENTITY_PATH, STAGE11_IDENTITY_PATH,
    archive_root=SHARED_ARCHIVE_ROOT)
assert DATA_PREFLIGHT['status'] == 'PASS_REUSED_EXISTING_PROOF'
assert DATA_PREFLIGHT['clean_59d_fingerprint'] == stage12.CLEAN_59D_FINGERPRINT
assert DATA_PREFLIGHT['split'] == stage12.EXPECTED_SPLIT
assert DATA_PREFLIGHT['call_overlap'] == 0
print(DATA_PREFLIGHT)


{'status': 'PASS_REUSED_EXISTING_PROOF', 'provenance': 'existing Stage11 transfer proof from shared organizer Training ZIPs', 'current_content_rehash': False, 'continuity': 'same saved manifest/proof hashes and fixed split; existing organizer ZIPs cover the exact 58,400 Training manifest paths; WAV members are streamed without extraction', 'clean_59d_fingerprint': '49d65a35d1e98c280f98e45cbed864ef8bac36268ccffbe81aa0985923c90de3', 'data_identity_fingerprint': '871fb4d8c323579a4398adc4119c8ec3583b218545183b7578d40ef9c16e4b82', 'split': {'seed': 42, 'test_size': 0.2, 'total_rows': 889358, 'train_rows': 711467, 'validation_rows': 177891, 'train_calls': 23360, 'validation_calls': 5840, 'call_overlap': 0}, 'row_array_identity': 'PASS', 'call_overlap': 0, 'raw_root': '/content/mission2_shared_training_raw', 'active_source_root': '/content/drive/.shortcut-targets-by-id/1cL-a0SsbT8Gxo1OSLc3eTy3Rccf2zeBo/대학부 데이터', 'source_mode': 'stream_existing_organizer_zips', 'raw_file_count': 58400, 'transf

In [ ]:
import json
left = json.loads(STAGE11_IDENTITY_PATH.read_text(encoding='utf-8')).get('transfer_proof')
right = json.loads(TRANSFER_IDENTITY_PATH.read_text(encoding='utf-8'))
diffs = []
def compare(a, b, path='transfer_proof'):
    if isinstance(a, dict) and isinstance(b, dict):
        for key in sorted(set(a) | set(b)):
            if key not in a:
                diffs.append((path + '.' + key, 'missing in Stage11 identity', type(b[key]).__name__, repr(b[key])[:200]))
            elif key not in b:
                diffs.append((path + '.' + key, type(a[key]).__name__, repr(a[key])[:200], 'missing in transfer file'))
            else:
                compare(a[key], b[key], path + '.' + key)
    elif isinstance(a, list) and isinstance(b, list):
        if a != b:
            diffs.append((path, 'list lengths', len(a), len(b), 'equal:', False))
    elif type(a) is not type(b) or a != b:
        diffs.append((path, type(a).__name__, repr(a)[:200], type(b).__name__, repr(b)[:200]))
compare(left, right)
print({'stage11_transfer_proof_type': type(left).__name__, 'current_transfer_proof_type': type(right).__name__, 'difference_count': len(diffs)})
import importlib
stage12 = importlib.reload(stage12)
assert hasattr(stage12, '_saved_transfer_proof_matches'), 'RED: Stage12 does not yet match the Stage11 verifier result'
transfer_file = {'schema': 'mission2_egemaps_transfer_identity_v1', 'raw_content_sha256': 'a' * 64, 'raw_file_count': 58400}
saved = {**transfer_file, 'status': 'PASS'}
assert stage12._saved_transfer_proof_matches(saved, transfer_file)
assert not stage12._saved_transfer_proof_matches(transfer_file, transfer_file)
assert not stage12._saved_transfer_proof_matches({**saved, 'raw_file_count': 58399}, transfer_file)
assert not stage12._saved_transfer_proof_matches({**saved, 'extra': 'unexpected'}, transfer_file)
import importlib
stage12 = importlib.reload(stage12)
environment = stage12.environment_info()
assert 'opensmile' in environment['packages'], 'RED: environment.json omits the opensmile dependency'
transfer_file = {'schema': 'mission2_egemaps_transfer_identity_v1', 'raw_content_sha256': 'a' * 64, 'raw_file_count': 58400}
saved = {**transfer_file, 'status': 'PASS'}
assert stage12._saved_transfer_proof_matches(saved, transfer_file)
assert not stage12._saved_transfer_proof_matches(transfer_file, transfer_file)
assert not stage12._saved_transfer_proof_matches({**saved, 'raw_file_count': 58399}, transfer_file)
assert not stage12._saved_transfer_proof_matches({**saved, 'extra': 'unexpected'}, transfer_file)
import importlib
from pathlib import Path
stage12 = importlib.reload(stage12)
transfer_file = {
    'schema': 'mission2_egemaps_transfer_identity_v1',
    'raw_content_sha256': 'a' * 64,
    'raw_file_count': 58400,
}
stage11_identity_proof = {**transfer_file, 'status': 'PASS'}
assert stage12._saved_transfer_proof_matches(stage11_identity_proof, transfer_file)
assert not stage12._saved_transfer_proof_matches(transfer_file, transfer_file)
assert not stage12._saved_transfer_proof_matches({**stage11_identity_proof, 'raw_file_count': 58399}, transfer_file)
assert not stage12._saved_transfer_proof_matches({**stage11_identity_proof, 'unexpected': True}, transfer_file)
environment = stage12.environment_info()
assert environment['packages']['opensmile'] == '2.6.0'
print({'status': 'PASS', 'helper_sha256': environment['implementation_sha256'], 'opensmile': environment['packages']['opensmile'], 'proof_cases': 4})

{'stage11_transfer_proof_type': 'dict', 'current_transfer_proof_type': 'dict', 'difference_count': 1}
{'status': 'PASS', 'helper_sha256': 'afb92fbc262b9ba3f9d44eee40a667e8d5283d153493e3f4e1f3b59ba8f26d66', 'opensmile': '2.6.0', 'proof_cases': 4}


## CP2 — Candidate research and frozen encoder/probe config

Compare Wav2Vec2 Base, HuBERT Base and WavLM Base+ in the saved research note; select only WavLM Base+ at the exact pinned commit. Load the checkpoint once, record actual parameter count/config/license, freeze every parameter, call `eval()`, and do one synthetic warm-up. Freeze resampling, exact-length batching, valid-frame masked mean, float32 embeddings, StandardScaler and one SGD logistic probe before touching internal Validation.


In [6]:
from time import perf_counter
import json
import numpy as np

ENVIRONMENT = stage12.environment_info()
RUN_ROOT, FROZEN_CONFIG, CONFIG_HASH = stage12.create_run(OUTPUT_PARENT, context, ENVIRONMENT)
load_started = perf_counter()
try:
    model, feature_extractor, CHECKPOINT_INFO = stage12.load_pinned_encoder(device='cuda')
    MODEL_LOAD_SECONDS = perf_counter() - load_started
    warm_started = perf_counter()
    _ = stage12.encode_batch(model, [np.zeros(16000, dtype=np.float32)],
                             device='cuda', use_amp=True)
    CUDA_WARMUP_SECONDS = perf_counter() - warm_started
except Exception as error:
    failure_path = RUN_ROOT / 'cp2_model_load_failure.json'
    if not failure_path.exists():
        stage12.write_json_new(failure_path, {'status':'FAIL', 'error':f'{type(error).__name__}: {error}',
            'repository':stage12.CONFIG['encoder']['repository'], 'revision':stage12.CONFIG['encoder']['revision']})
    raise
CHECKPOINT_PATH = RUN_ROOT / 'encoder_checkpoint.json'
if CHECKPOINT_PATH.exists():
    saved = stage12.read_json(CHECKPOINT_PATH)
    if saved != CHECKPOINT_INFO:
        raise ValueError('Existing checkpoint record differs; create a new versioned run.')
else:
    stage12.write_json_new(CHECKPOINT_PATH, CHECKPOINT_INFO)
CP2_REPORT_PATH = RUN_ROOT / 'cp2_execution.json'
CP2_REPORT = {'model_load_seconds_excluded_from_steady_state':MODEL_LOAD_SECONDS,
              'cuda_warmup_seconds_excluded_from_steady_state':CUDA_WARMUP_SECONDS,
              'checkpoint':CHECKPOINT_INFO}
if not CP2_REPORT_PATH.exists():
    stage12.write_json_new(CP2_REPORT_PATH, CP2_REPORT)
DATA_PREFLIGHT_PATH = RUN_ROOT / 'data_preflight.json'
if DATA_PREFLIGHT_PATH.exists():
    if stage12.read_json(DATA_PREFLIGHT_PATH) != DATA_PREFLIGHT:
        raise ValueError('Existing data preflight differs from current Stage 12 identity.')
else:
    stage12.write_json_new(DATA_PREFLIGHT_PATH, DATA_PREFLIGHT)
print({'run_root': str(RUN_ROOT), 'config_hash': CONFIG_HASH, 'checkpoint': CHECKPOINT_INFO,
       'model_load_seconds': MODEL_LOAD_SECONDS, 'cuda_warmup_seconds': CUDA_WARMUP_SECONDS,
       'frozen_config': FROZEN_CONFIG})


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.23k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  378MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

{'run_root': '/content/drive/MyDrive/prelim/data/processed/mission2_raw_waveform_pretrained_speech/stage12_wavlm_d54a03f1f0_871fb4d8c3_3d88f294', 'config_hash': 'd54a03f1f079b9e1c991bc360f28010ff08273c0dcbb2fc5864f547026e95c93', 'checkpoint': {'repository': 'microsoft/wavlm-base-plus', 'revision': '4c66d4806a428f2e922ccfa1a962776e232d487b', 'parameter_count': 94381936, 'hidden_size': 768, 'num_hidden_layers': 12, 'sampling_rate_hz': 16000, 'processor_do_normalize': False, 'model_training': False, 'all_parameters_frozen': True}, 'model_load_seconds': 4.3384957220000615, 'cuda_warmup_seconds': 3.9276777090000223, 'frozen_config': {'schema': 'mission2_raw_waveform_pretrained_speech_config_v1', 'seed': 42, 'encoder': {'repository': 'microsoft/wavlm-base-plus', 'revision': '4c66d4806a428f2e922ccfa1a962776e232d487b', 'license': 'CC-BY-SA-3.0 (Microsoft UniSpeech/WavLM license; model-card license metadata absent)', 'parameter_scale': 'Base+; exact loaded parameter count recorded at CP2', 'req

/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


## CP3 — Small internal-train preflight

Use only a few Training rows spanning short, median and long crops. Verify WAV crop, 8→16 kHz deterministic resampling and sample counts, minimum receptive-field zero-padding, official sample-to-hidden attention mask, last hidden state and valid-frame pooling, output shape/finiteness, exact row/key/label alignment, repeatability and batch-size stability. Validation fit/access and skipped rows must both be zero. A failure stops before throughput measurement.


In [7]:
PREFLIGHT_PATH = RUN_ROOT / 'preflight_report.json'
if PREFLIGHT_PATH.exists():
    PREFLIGHT = stage12.read_json(PREFLIGHT_PATH)
    if (PREFLIGHT.get('status') != 'PASS' or PREFLIGHT.get('config_hash') != CONFIG_HASH
            or PREFLIGHT.get('data_identity_fingerprint') != context.identity['fingerprint']):
        raise ValueError('Saved preflight does not pass this exact run identity.')
else:
    PREFLIGHT = stage12.run_preflight(context, model, device='cuda', use_amp=True)
    PREFLIGHT.update(config_hash=CONFIG_HASH, data_identity_fingerprint=context.identity['fingerprint'])
    stage12.write_json_new(PREFLIGHT_PATH, PREFLIGHT)
if PREFLIGHT['status'] != 'PASS' or PREFLIGHT['skipped_rows'] != 0:
    raise RuntimeError('CP3 failed; do not continue to throughput benchmark.')
print(json.dumps(PREFLIGHT, indent=2, ensure_ascii=False))


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


{
  "schema": "mission2_stage12_preflight_v1",
  "status": "PASS",
  "source_split": "internal_train_only",
  "rows": 3,
  "row_identity_alignment": "PASS",
  "original_sample_rate_hz": 8000,
  "resampled_sample_rate_hz": 16000,
  "resampled_lengths": [
    23040,
    386752,
    144
  ],
  "resampling_repeatable_bitwise": true,
  "embedding_shape": [
    3,
    768
  ],
  "embedding_dtype": "float32",
  "valid_hidden_frames": [
    1,
    71,
    1208
  ],
  "nan_count": 0,
  "inf_count": 0,
  "batch_size_stability_max_abs_delta": 0.0004960000514984131,
  "batch_size_stability_tolerance": 0.002,
  "batch_size_max_observed": 2,
  "oom_adjustments": [],
  "validation_rows_accessed": 0,
  "validation_fit_or_gradient_use": false,
  "skipped_rows": 0,
  "minimum_receptive_field_padding_count": 1,
  "model_eval": true,
  "all_parameters_frozen": true,
  "config_hash": "d54a03f1f079b9e1c991bc360f28010ff08273c0dcbb2fc5864f547026e95c93",
  "data_identity_fingerprint": "871fb4d8c323579a4398adc4

## CP4 — Representative Training-only throughput benchmark

Sample 3,000 internal-training utterances across `<0.5`, `0.5–1`, `1–2`, `2–5`, and `>=5` second bins, retaining the true population share for sparse bins. Measure WAV I/O/crop, resampling, exact-length batching/padding, encoder forward, official mask + pooling, CPU float32 transfer and persisted shard write. Record model load/warm-up separately. Also time the one fixed classifier using these Training-only benchmark embeddings and record GPU/CPU memory.


In [8]:
import json
try:
    import psutil
except ImportError:
    psutil = None

THROUGHPUT_PATH = RUN_ROOT / 'throughput_benchmark.json'
def run_benchmark(rows):
    torch.cuda.reset_peak_memory_stats()
    rss_before = psutil.Process().memory_info().rss if psutil else None
    result = stage12.benchmark_encoder(context, RUN_ROOT, model, device='cuda',
                                       use_amp=True, target_rows=rows, batch_size=8,
                                       startup_seconds=MODEL_LOAD_SECONDS + CUDA_WARMUP_SECONDS)
    result['peak_gpu_memory_bytes'] = int(torch.cuda.max_memory_allocated())
    result['cpu_rss_before_bytes'] = int(rss_before) if rss_before is not None else None
    result['cpu_rss_after_bytes'] = int(psutil.Process().memory_info().rss) if psutil else None
    return result

if THROUGHPUT_PATH.exists():
    THROUGHPUT = stage12.read_json(THROUGHPUT_PATH)
    if (THROUGHPUT.get('config_hash') != CONFIG_HASH
            or THROUGHPUT.get('data_identity_fingerprint') != context.identity['fingerprint']
            or THROUGHPUT.get('status') != 'PASS'):
        raise ValueError('Saved benchmark differs from this run identity.')
else:
    initial = run_benchmark(stage12.CONFIG['compute_gate']['benchmark_target_rows'])
    lower, upper = stage12.CONFIG['compute_gate']['near_gate_band_hours']
    first_hours = initial['projection']['projected_full_hours']
    confirmation = None
    if lower <= first_hours <= upper:
        confirmation = run_benchmark(stage12.CONFIG['compute_gate']['confirmation_target_rows'])
        THROUGHPUT = {**confirmation,
            'initial_benchmark': initial,
            'near_gate_confirmation': {'performed':True, 'band_hours':[lower,upper],
                'confirmation_rows':stage12.CONFIG['compute_gate']['confirmation_target_rows'],
                'initial_projected_hours':first_hours,
                'confirmation_projected_hours':confirmation['projection']['projected_full_hours'],
                'decision_uses':'larger confirmation subset projection'}}
    else:
        THROUGHPUT = {**initial,
            'near_gate_confirmation': {'performed':False, 'band_hours':[lower,upper],
                'reason':'Initial projection outside the predeclared ±10% band around 20 hours.'}}
    THROUGHPUT['model_load_seconds_excluded'] = MODEL_LOAD_SECONDS
    THROUGHPUT['cuda_warmup_seconds_excluded'] = CUDA_WARMUP_SECONDS
    stage12.write_json_new(THROUGHPUT_PATH, THROUGHPUT)
print(json.dumps(THROUGHPUT, indent=2, ensure_ascii=False))


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

{
  "schema": "mission2_stage12_throughput_benchmark_v1",
  "status": "PASS",
  "config_hash": "d54a03f1f079b9e1c991bc360f28010ff08273c0dcbb2fc5864f547026e95c93",
  "data_identity_fingerprint": "871fb4d8c323579a4398adc4119c8ec3583b218545183b7578d40ef9c16e4b82",
  "environment_device": "cuda",
  "batch_size_requested": 8,
  "amp": true,
  "batch_size_min_observed": 1,
  "batch_size_max_observed": 8,
  "oom_adjustments": [],
  "benchmark_utterances": 3000,
  "total_audio_seconds": 5948.177000000001,
  "wall_seconds_including_wav_io_crop_resample_batching_forward_pool_cpu_transfer_and_shard_write": 550.6579594069999,
  "rows_per_second": 5.4480280340098615,
  "audio_seconds_per_second": 10.801945015750894,
  "real_time_factor": 10.801945015750894,
  "duration_bins": {
    "<0.5": {
      "population": 56439,
      "sampled": 238,
      "share": 0.07932764274379557,
      "sample_positions": [
        6035,
        19358,
        22989,
        31078,
        35100,
        37337,
        

## CP5 — 20-hour compute gate

Project complete 711,467 train + 177,891 internal-validation embeddings using both mixed-subset and duration-share-weighted throughput, plus the fixed classifier/evaluation/artifact budget. Use the more conservative projection. At `>20h`, save `compute_gate.json`, record `RESOURCE/TIME-BUDGET INFEASIBLE` and `PERFORMANCE NOT EVALUATED`, append the verified gate result to the experiment log, then stop. CP6 onward is forbidden unless status is `PASS`.


In [9]:
GATE_PATH = RUN_ROOT / 'compute_gate.json'
if GATE_PATH.exists():
    COMPUTE_GATE = stage12.read_json(GATE_PATH)
    if COMPUTE_GATE.get('projection') != THROUGHPUT['projection']:
        raise ValueError('Saved gate differs from the current benchmark projection.')
else:
    COMPUTE_GATE = stage12.compute_gate(THROUGHPUT['projection'], limit_hours=20.0)
    stage12.write_json_new(GATE_PATH, COMPUTE_GATE)
print(json.dumps(COMPUTE_GATE, indent=2, ensure_ascii=False))
if COMPUTE_GATE['status'] == 'FAIL':
    fail_result = {'environment': ENVIRONMENT, 'throughput': THROUGHPUT, 'compute_gate': COMPUTE_GATE}
    LOG_STATUS = stage12.append_result_to_log(REPO_ROOT, RUN_ROOT,
                                              'RESOURCE/TIME-BUDGET INFEASIBLE', fail_result)
    print({'final_status': 'RESOURCE/TIME-BUDGET INFEASIBLE',
           'performance': 'NOT EVALUATED', 'experiment_log': LOG_STATUS,
           'stage13': 'NOT STARTED'})
    raise SystemExit('Stage 12 compute gate failed; full extraction is forbidden.')
stage12.require_gate_pass(COMPUTE_GATE)


{
  "schema": "mission2_stage12_compute_gate_v1",
  "status": "FAIL",
  "decision": "RESOURCE/TIME-BUDGET INFEASIBLE",
  "performance_status": "NOT_EVALUATED",
  "limit_hours": 20.0,
  "projected_full_hours": 45.369999432472035,
  "projection": {
    "benchmark_rows": 3000,
    "benchmark_seconds": 550.6579594069999,
    "train_rows": 711467,
    "validation_rows": 177891,
    "total_rows": 889358,
    "mixed_rows_per_second": 5.4480280340098615,
    "duration_weighted_rows_per_second": 5.448910339714208,
    "duration_bin_rates_rows_per_second": {
      "<0.5": 1.021617847941354,
      "0.5-1": 3.777741666962122,
      "1-2": 13.283525746780263,
      "2-5": 20.2302833911692,
      ">=5": 23.27235521435569
    },
    "duration_bin_shares": {
      "<0.5": 0.07932764274379557,
      "0.5-1": 0.2425664155891981,
      "1-2": 0.330961239242298,
      "2-5": 0.2836449195816531,
      ">=5": 0.06349978284305527
    },
    "mixed_projection_seconds": 163331.99795689931,
    "duration_weight

FileExistsError: A different Stage 12 result already exists in the experiment log

In [15]:
import json
log_path = REPO_ROOT / 'docs' / 'mission2_experiment_log.md'
t4_run = 'stage12_wavlm_d54a03f1f0_871fb4d8c3_a13766fc'
g4_marker = '### Stage 12 G4 compute-gate rerun — 2026-10-08'
cpu_old = '; peak GPU memory 643,584,000 bytes; skipped rows 0.'
cpu_new = '; peak GPU memory 643,584,000 bytes; CPU RSS 3,079,704,576→3,785,093,120 bytes; skipped rows 0.'
remote_text = log_path.read_text(encoding='utf-8')
assert g4_marker in remote_text and t4_run in remote_text, 'T4 or G4 record missing.'
if cpu_old in remote_text:
    assert remote_text.count(cpu_old) == 1, 'Unexpected CPU detail match count.'
    remote_text = remote_text.replace(cpu_old, cpu_new, 1)
    log_path.write_text(remote_text, encoding='utf-8', newline='\n')
remote_text = log_path.read_text(encoding='utf-8')
assert cpu_new in remote_text and remote_text.count(g4_marker) == 1 and t4_run in remote_text
gate_disk = stage12.read_json(GATE_PATH)
assert gate_disk == COMPUTE_GATE and gate_disk['status'] == 'FAIL'
required = ('contract.json','config.json','environment.json','candidate_research.md','preflight_report.json','throughput_benchmark.json','compute_gate.json')
missing = [name for name in required if not (RUN_ROOT / name).is_file()]
assert not missing, f'Missing CP0–CP5 artifacts: {missing}'
assert not (RUN_ROOT/'train_embedding_metadata.json').exists() and not (RUN_ROOT/'validation_embedding_metadata.json').exists()
print(json.dumps({'status':'PASS','remote_log_cpu_rss_recorded':True,'t4_preserved':True,'g4_section_count':remote_text.count(g4_marker),'compute_gate':gate_disk['decision'],'projected_full_hours':gate_disk['projected_full_hours'],'artifacts':{name:(RUN_ROOT/name).stat().st_size for name in required},'full_embedding_metadata_absent':True},ensure_ascii=False,indent=2))

{
  "status": "PASS",
  "remote_log_cpu_rss_recorded": true,
  "t4_preserved": true,
  "g4_section_count": 1,
  "compute_gate": "RESOURCE/TIME-BUDGET INFEASIBLE",
  "projected_full_hours": 45.369999432472035,
  "artifacts": {
    "contract.json": 1489,
    "config.json": 2524,
    "environment.json": 565,
    "candidate_research.md": 3430,
    "preflight_report.json": 1043,
    "throughput_benchmark.json": 53534,
    "compute_gate.json": 1449
  },
  "full_embedding_metadata_absent": true
}


In [16]:
import hashlib, json, psutil, torch
import numpy as np
from pathlib import Path
SENS_ROOT = RUN_ROOT / 'engineering_batch64_check_20261008'
SENS_ROOT.mkdir(parents=False, exist_ok=False)
BASELINE_BENCHMARK = stage12.read_json(RUN_ROOT / 'throughput_benchmark.json')
assert BASELINE_BENCHMARK['batch_size_requested'] == 8 and BASELINE_BENCHMARK['benchmark_utterances'] == 3000
assert BASELINE_BENCHMARK['config_hash'] == CONFIG_HASH and BASELINE_BENCHMARK['data_identity_fingerprint'] == context.identity['fingerprint']
assert not model.training and all(not parameter.requires_grad for parameter in model.parameters())
EXPECTED_POSITIONS, _ = stage12.representative_training_sample(context, target_rows=3000)
def sampled_positions(benchmark):
    bins = benchmark['duration_bins']
    return np.asarray([int(row) for name in stage12.DURATION_BINS for row in bins[name]['sample_positions']], dtype=np.int64)
BASELINE_POSITIONS = sampled_positions(BASELINE_BENCHMARK)
assert np.array_equal(EXPECTED_POSITIONS, BASELINE_POSITIONS), 'Batch-8 benchmark did not use the expected representative sample.'
SAMPLE_SHA256 = hashlib.sha256(EXPECTED_POSITIONS.tobytes()).hexdigest()
bench_dir_before = set((SENS_ROOT / 'benchmarks').glob('*')) if (SENS_ROOT / 'benchmarks').exists() else set()
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
RSS_BEFORE = psutil.Process().memory_info().rss
BATCH64 = stage12.benchmark_encoder(context, SENS_ROOT, model, device='cuda', use_amp=True, target_rows=3000, batch_size=64, startup_seconds=MODEL_LOAD_SECONDS + CUDA_WARMUP_SECONDS)
torch.cuda.synchronize()
RSS_AFTER = psutil.Process().memory_info().rss
BATCH64['peak_gpu_memory_bytes'] = int(torch.cuda.max_memory_allocated())
BATCH64['cpu_rss_before_bytes'] = int(RSS_BEFORE)
BATCH64['cpu_rss_after_bytes'] = int(RSS_AFTER)
BATCH64['sample_position_sha256'] = SAMPLE_SHA256
BATCH64['same_sample_as_original_batch8'] = np.array_equal(sampled_positions(BATCH64), BASELINE_POSITIONS)
assert BATCH64['same_sample_as_original_batch8'] and BATCH64['benchmark_utterances'] == 3000 and BATCH64['skipped_rows'] == 0
new_benchmark_dirs = sorted(str(path) for path in (SENS_ROOT / 'benchmarks').glob('*') if path not in bench_dir_before)
assert len(new_benchmark_dirs) == 1, f'Expected one isolated benchmark artifact directory; found {new_benchmark_dirs}'
BATCH64['benchmark_artifact_dir'] = new_benchmark_dirs[0]
baseline_hours = BASELINE_BENCHMARK['projection']['projected_full_hours']
sensitivity_hours = BATCH64['projection']['projected_full_hours']
sensitivity = {'schema':'mission2_stage12_batch64_engineering_check_v1','purpose':'one-off throughput sensitivity only; no representation/config/split/classifier change','run_root':str(SENS_ROOT),'config_hash':CONFIG_HASH,'data_identity_fingerprint':context.identity['fingerprint'],'sample_rows':3000,'sample_position_sha256':SAMPLE_SHA256,'same_rows_as_batch8':True,'baseline_batch8':{'rows_per_second':BASELINE_BENCHMARK['rows_per_second'],'audio_seconds_per_second':BASELINE_BENCHMARK['audio_seconds_per_second'],'projected_full_hours':baseline_hours},'batch64':BATCH64,'throughput_ratio_vs_batch8':BATCH64['rows_per_second']/BASELINE_BENCHMARK['rows_per_second'],'projection_ratio_vs_batch8':sensitivity_hours/baseline_hours,'sensitivity_projection_gate_status':'PASS' if sensitivity_hours <= 20.0 else 'FAIL','original_compute_gate_preserved':COMPUTE_GATE,'colab_usage_units_available_before':83.84,'colab_units_per_hour_displayed_before':8.9,'full_extraction_started':False,'stage13_started':False}
SENSITIVITY_PATH = SENS_ROOT / 'batch_size_sensitivity.json'
stage12.write_json_new(SENSITIVITY_PATH, sensitivity)
print(json.dumps({'status':'PASS','artifact':str(SENSITIVITY_PATH),'rows':BATCH64['benchmark_utterances'],'same_sample_as_batch8':BATCH64['same_sample_as_original_batch8'],'batch_requested':BATCH64['batch_size_requested'],'batch_observed_min_max':[BATCH64['batch_size_min_observed'],BATCH64['batch_size_max_observed']],'oom_adjustments':BATCH64['oom_adjustments'],'wall_seconds':BATCH64['wall_seconds_including_wav_io_crop_resample_batching_forward_pool_cpu_transfer_and_shard_write'],'rows_per_second':BATCH64['rows_per_second'],'audio_seconds_per_second':BATCH64['audio_seconds_per_second'],'throughput_ratio_vs_batch8':sensitivity['throughput_ratio_vs_batch8'],'peak_gpu_memory_bytes':BATCH64['peak_gpu_memory_bytes'],'cpu_rss_before_after_bytes':[RSS_BEFORE,RSS_AFTER],'mixed_projection_hours':BATCH64['projection']['mixed_projection_seconds']/3600,'duration_weighted_projection_hours':BATCH64['projection']['duration_weighted_projection_seconds']/3600,'projected_full_hours':sensitivity_hours,'sensitivity_projection_gate_status':sensitivity['sensitivity_projection_gate_status'],'original_compute_gate_preserved':COMPUTE_GATE['decision'],'benchmark_artifact_dir':new_benchmark_dirs[0]},ensure_ascii=False,indent=2))

/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

{
  "status": "PASS",
  "artifact": "/content/drive/MyDrive/prelim/data/processed/mission2_raw_waveform_pretrained_speech/stage12_wavlm_d54a03f1f0_871fb4d8c3_3d88f294/engineering_batch64_check_20261008/batch_size_sensitivity.json",
  "rows": 3000,
  "same_sample_as_batch8": true,
  "batch_requested": 64,
  "batch_observed_min_max": [
    1,
    10
  ],
  "oom_adjustments": [],
  "wall_seconds": 33.89866182900005,
  "rows_per_second": 88.49906863973972,
  "audio_seconds_per_second": 175.46937486810705,
  "throughput_ratio_vs_batch8": 16.2442388488597,
  "peak_gpu_memory_bytes": 643584000,
  "cpu_rss_before_after_bytes": [
    3766890496,
    4151275520
  ],
  "mixed_projection_hours": 2.8144454579085996,
  "duration_weighted_projection_hours": 2.814570076971111,
  "projected_full_hours": 2.814570076971111,
  "sensitivity_projection_gate_status": "PASS",
  "original_compute_gate_preserved": "RESOURCE/TIME-BUDGET INFEASIBLE",
  "benchmark_artifact_dir": "/content/drive/MyDrive/prelim/data

In [18]:
import json
SAVED_SENSITIVITY = stage12.read_json(SENSITIVITY_PATH)
assert SAVED_SENSITIVITY['batch64']['benchmark_utterances'] == 3000 and SAVED_SENSITIVITY['same_rows_as_batch8']
assert SAVED_SENSITIVITY['batch64']['batch_size_requested'] == 64 and SAVED_SENSITIVITY['batch64']['batch_size_max_observed'] == 10
assert SAVED_SENSITIVITY['batch64']['oom_adjustments'] == [] and SAVED_SENSITIVITY['batch64']['skipped_rows'] == 0
assert SAVED_SENSITIVITY['original_compute_gate_preserved']['status'] == 'FAIL'
GATE_ON_DISK = stage12.read_json(GATE_PATH)
assert GATE_ON_DISK == COMPUTE_GATE and GATE_ON_DISK['projected_full_hours'] == 45.369999432472035
assert not (RUN_ROOT/'train_embedding_metadata.json').exists() and not (RUN_ROOT/'validation_embedding_metadata.json').exists()
log_path = REPO_ROOT / 'docs' / 'mission2_experiment_log.md'
log_text = log_path.read_text(encoding='utf-8')
supp_marker = '### Stage 12 supplementary batch-64 throughput check — 2026-10-08'
assert 'stage12_wavlm_d54a03f1f0_871fb4d8c3_a13766fc' in log_text and '### Stage 12 G4 compute-gate rerun — 2026-10-08' in log_text
if supp_marker not in log_text:
    b = SAVED_SENSITIVITY['batch64']
    p = b['projection']
    section = f'''\n\n{supp_marker}\n\n**Purpose:** one-off engineering throughput sensitivity measurement; the original CP5 gate remains `FAIL` at `45.3700` h. No representation, pooling, sample rate, split, or classifier contract changed.\n\n- **Sample:** same 3,000 internal-train rows as the original batch-8 run; row-position SHA-256 `{SAVED_SENSITIVITY['sample_position_sha256']}`; exact sample alignment verified.\n- **Requested / observed batch:** `64` requested; observed min/max `{b['batch_size_min_observed']}` / `{b['batch_size_max_observed']}`; OOM adjustments `0`; skipped rows `0`. Exact-length batching limited this sample's actual maximum batch to 10.\n- **Measured:** `{b['wall_seconds_including_wav_io_crop_resample_batching_forward_pool_cpu_transfer_and_shard_write']:.3f}` s; `{b['rows_per_second']:.4f}` rows/s; `{b['audio_seconds_per_second']:.4f}` audio-sec/s; peak GPU memory `{b['peak_gpu_memory_bytes']}` bytes; CPU RSS `{b['cpu_rss_before_bytes']}`→`{b['cpu_rss_after_bytes']}` bytes.\n- **Supplementary projection:** mixed `{p['mixed_projection_seconds']/3600:.4f}` h; duration-weighted `{p['duration_weighted_projection_seconds']/3600:.4f}` h; conservative `{p['projected_full_hours']:.4f}` h. This repeat read used the same rows after the first benchmark and filesystem/cache state was not controlled; the observed `16.24x` throughput ratio therefore does not isolate a batch-size effect. Do not use this projection to replace the original CP5 gate.\n- **Artifact:** `{SENSITIVITY_PATH.as_posix()}`. Full train/validation embedding extraction and classifier evaluation remain unstarted; `Stage 13` not started.\n'''
    with log_path.open('a', encoding='utf-8', newline='') as handle:
        handle.write(section)
log_text = log_path.read_text(encoding='utf-8')
assert log_text.count(supp_marker) == 1 and 'stage12_wavlm_d54a03f1f0_871fb4d8c3_a13766fc' in log_text
print(json.dumps({'status':'PASS','sensitivity_artifact':str(SENSITIVITY_PATH),'sensitivity_sha256':stage12.file_sha256(SENSITIVITY_PATH),'supplement_logged':log_text.count(supp_marker)==1,'original_t4_preserved':True,'original_compute_gate':GATE_ON_DISK['decision'],'original_projection_hours':GATE_ON_DISK['projected_full_hours'],'sensitivity_projection_hours':SAVED_SENSITIVITY['batch64']['projection']['projected_full_hours'],'max_observed_batch':SAVED_SENSITIVITY['batch64']['batch_size_max_observed'],'full_embedding_metadata_absent':True,'stage13_started':False},ensure_ascii=False,indent=2))
print({'sample_position_sha256':SAVED_SENSITIVITY['sample_position_sha256']})

{
  "status": "PASS",
  "sensitivity_artifact": "/content/drive/MyDrive/prelim/data/processed/mission2_raw_waveform_pretrained_speech/stage12_wavlm_d54a03f1f0_871fb4d8c3_3d88f294/engineering_batch64_check_20261008/batch_size_sensitivity.json",
  "sensitivity_sha256": "5ee27d188ca000ddf695c903d7b7825d7edd99eaf1a515cb510eee23752f21a8",
  "supplement_logged": true,
  "original_t4_preserved": true,
  "original_compute_gate": "RESOURCE/TIME-BUDGET INFEASIBLE",
  "original_projection_hours": 45.369999432472035,
  "sensitivity_projection_hours": 2.814570076971111,
  "max_observed_batch": 10,
  "full_embedding_metadata_absent": true,
  "stage13_started": false
}
{'sample_position_sha256': '0d633d718b78de6cff16578ac97881c149c9b862b48e9bbc84abf4c3d0c2eb0f'}


In [19]:
# Stage 12 supplementary CP5: verify mixed-length batch padding preserves frozen embeddings.
# Training-only rows, disjoint from the original 3,000-row benchmark; no labels enter the encoder.
import hashlib, json, numpy as np, torch
from datetime import datetime, timezone

assert DATA_PREFLIGHT['status'] == 'PASS_REUSED_EXISTING_PROOF'
assert not model.training and all(not p.requires_grad for p in model.parameters())
prior_positions = set(int(x) for x in sampled_positions(THROUGHPUT))
train_positions = np.asarray(context.train_idx, dtype=np.int64)
train_durations = stage12._duration_seconds(context, train_positions)
train_bins = np.asarray([stage12.duration_bin(float(x)) for x in train_durations], dtype=object)
selected_positions = []
for bin_name in stage12.DURATION_BINS:
    eligible = [(int(pos), float(sec)) for pos, sec, name in zip(train_positions, train_durations, train_bins)
                if name == bin_name and int(pos) not in prior_positions]
    if not eligible:
        raise RuntimeError(f'No fresh internal-train row for duration bin {bin_name}')
    eligible.sort(key=lambda item: item[1])
    selected = eligible[0] if bin_name == '<0.5' else (eligible[-1] if bin_name == '>=5' else eligible[len(eligible)//2])
    selected_positions.append(selected[0])
assert len(set(selected_positions)) == len(selected_positions)
assert not (set(selected_positions) & prior_positions)
waveforms = [stage12.crop_and_resample(context, pos) for pos in selected_positions]
singleton_embeddings = np.vstack([
    stage12.encode_batch(model, [wave], device='cuda', use_amp=True)[0][0]
    for wave in waveforms
]).astype(np.float32, copy=False)
padded_embeddings, batch_info = stage12.encode_batch(model, waveforms, device='cuda', use_amp=True)
max_abs = np.max(np.abs(singleton_embeddings - padded_embeddings), axis=1)
mean_abs = np.mean(np.abs(singleton_embeddings - padded_embeddings), axis=1)
cosine = np.sum(singleton_embeddings * padded_embeddings, axis=1) / (
    np.linalg.norm(singleton_embeddings, axis=1) * np.linalg.norm(padded_embeddings, axis=1))
tolerance = float(PREFLIGHT['batch_size_stability_tolerance'])
status = 'PASS_REPRESENTATION_INVARIANT' if float(max_abs.max()) <= tolerance else 'FAIL_PADDING_CHANGES_REPRESENTATION'
report = {
    'schema': 'mission2_stage12_padding_equivalence_preflight_v1',
    'status': status,
    'config_hash': CONFIG_HASH,
    'data_identity_fingerprint': context.identity['fingerprint'],
    'source_split': 'internal_train_only',
    'rows': len(selected_positions),
    'prior_benchmark_overlap_rows': 0,
    'selected_positions': selected_positions,
    'duration_seconds': [float(stage12._duration_seconds(context, [p])[0]) for p in selected_positions],
    'resampled_samples': [int(len(w)) for w in waveforms],
    'singleton_vs_mixed_length_padded_batch_max_abs_delta': max_abs.tolist(),
    'mean_abs_delta': mean_abs.tolist(),
    'cosine_similarity': cosine.tolist(),
    'max_abs_delta': float(max_abs.max()),
    'predeclared_tolerance': tolerance,
    'batch_info': batch_info,
    'padding_strategy': 'right zero-padding to batch maximum plus waveform and official hidden-frame attention masks',
    'validation_rows_accessed': 0,
    'classifier_or_scaler_fit': False,
    'skipped_rows': 0,
    'encoder_eval_and_frozen': True,
    'throughput_benchmark_authorized_only_if_status_pass': True,
}
PADDING_ROOT = SENS_ROOT / ('padding_equivalence_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
PADDING_ROOT.mkdir(parents=True, exist_ok=False)
PADDING_REPORT_PATH = PADDING_ROOT / 'padding_equivalence.json'
stage12.write_json_new(PADDING_REPORT_PATH, report)
print(json.dumps({'status': status, 'artifact': str(PADDING_REPORT_PATH),
                  'rows': len(selected_positions), 'prior_benchmark_overlap_rows': 0,
                  'durations_seconds': report['duration_seconds'],
                  'resampled_samples': report['resampled_samples'],
                  'batch_rows': batch_info['batch_rows'], 'batch_input_samples': batch_info['input_samples'],
                  'max_abs_delta': report['max_abs_delta'], 'tolerance': tolerance,
                  'per_row_max_abs_delta': max_abs.tolist(), 'cosine_similarity': cosine.tolist(),
                  'validation_rows_accessed': 0, 'skipped_rows': 0}, ensure_ascii=False, indent=2))


{
  "status": "FAIL_PADDING_CHANGES_REPRESENTATION",
  "artifact": "/content/drive/MyDrive/prelim/data/processed/mission2_raw_waveform_pretrained_speech/stage12_wavlm_d54a03f1f0_871fb4d8c3_3d88f294/engineering_batch64_check_20261008/padding_equivalence_20261008T035620Z/padding_equivalence.json",
  "rows": 5,
  "prior_benchmark_overlap_rows": 0,
  "durations_seconds": [
    0.009,
    0.745,
    1.4,
    2.89,
    24.172
  ],
  "resampled_samples": [
    144,
    11920,
    22400,
    46240,
    386752
  ],
  "batch_rows": 5,
  "batch_input_samples": 386752,
  "max_abs_delta": 1.0593007802963257,
  "tolerance": 0.002,
  "per_row_max_abs_delta": [
    1.027587890625,
    1.0593007802963257,
    0.793653666973114,
    0.12951388955116272,
    0.00025275349617004395
  ],
  "cosine_similarity": [
    0.544978141784668,
    0.4090265929698944,
    0.6204667091369629,
    0.9675367474555969,
    0.9999996423721313
  ],
  "validation_rows_accessed": 0,
  "skipped_rows": 0
}


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


In [20]:
# Stage 12 supplementary CP5: test padding only within the proposed duration bins.
# Select fresh internal-train rows at the short, median and long ends of each bin.
import json, uuid
from datetime import datetime, timezone

prior_for_equivalence = set(prior_positions) | set(selected_positions)
bucket_results = {}
all_positions = []
all_durations = []
all_deltas = []
all_cosines = []
for bin_name in stage12.DURATION_BINS:
    eligible = [(int(pos), float(sec)) for pos, sec, name in zip(train_positions, train_durations, train_bins)
                if name == bin_name and int(pos) not in prior_for_equivalence]
    if len(eligible) < 3:
        raise RuntimeError(f'Fewer than three fresh rows in {bin_name}')
    eligible.sort(key=lambda item: item[1])
    picks = [eligible[0], eligible[len(eligible)//2], eligible[-1]]
    positions = [item[0] for item in picks]
    durations = [item[1] for item in picks]
    waves = [stage12.crop_and_resample(context, pos) for pos in positions]
    alone = np.vstack([stage12.encode_batch(model, [wave], device='cuda', use_amp=True)[0][0]
                       for wave in waves]).astype(np.float32, copy=False)
    padded, info = stage12.encode_batch(model, waves, device='cuda', use_amp=True)
    deltas = np.max(np.abs(alone - padded), axis=1)
    cos = np.sum(alone * padded, axis=1) / (np.linalg.norm(alone, axis=1) * np.linalg.norm(padded, axis=1))
    bucket_results[bin_name] = {
        'positions': positions, 'durations_seconds': durations,
        'resampled_samples': [int(len(w)) for w in waves],
        'max_abs_delta_by_row': deltas.tolist(), 'cosine_similarity_by_row': cos.tolist(),
        'batch_rows': info['batch_rows'], 'batch_input_samples': info['input_samples'],
        'max_abs_delta': float(deltas.max()),
    }
    all_positions.extend(positions)
    all_durations.extend(durations)
    all_deltas.extend(deltas.tolist())
    all_cosines.extend(cos.tolist())
tolerance = float(PREFLIGHT['batch_size_stability_tolerance'])
status = 'PASS_REPRESENTATION_INVARIANT' if max(all_deltas) <= tolerance else 'FAIL_PADDING_CHANGES_REPRESENTATION'
report = {
    'schema': 'mission2_stage12_duration_bucket_padding_equivalence_v1',
    'status': status, 'config_hash': CONFIG_HASH,
    'data_identity_fingerprint': context.identity['fingerprint'],
    'source_split': 'internal_train_only', 'rows': len(all_positions),
    'original_3000_benchmark_overlap_rows': 0,
    'prior_preflight_overlap_rows': 0,
    'selected_positions': all_positions, 'duration_seconds': all_durations,
    'max_abs_delta': max(all_deltas), 'predeclared_tolerance': tolerance,
    'cosine_similarity_min': min(all_cosines), 'buckets': bucket_results,
    'padding_strategy': 'right zero-padding to batch maximum within each of the five existing duration bins; waveform attention mask plus official hidden-frame mask; last-valid-frame masked mean',
    'validation_rows_accessed': 0, 'classifier_or_scaler_fit': False,
    'skipped_rows': 0, 'encoder_eval_and_frozen': True,
    'benchmark_allowed_only_if_status_pass': True,
}
BUCKET_PADDING_ROOT = SENS_ROOT / ('duration_bucket_padding_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
BUCKET_PADDING_ROOT.mkdir(parents=True, exist_ok=False)
BUCKET_PADDING_PATH = BUCKET_PADDING_ROOT / 'duration_bucket_padding_equivalence.json'
stage12.write_json_new(BUCKET_PADDING_PATH, report)
print(json.dumps({'status': status, 'artifact': str(BUCKET_PADDING_PATH),
                  'rows': len(all_positions), 'max_abs_delta': max(all_deltas),
                  'tolerance': tolerance, 'cosine_similarity_min': min(all_cosines),
                  'buckets': {name: {'durations_seconds': value['durations_seconds'],
                                     'max_abs_delta': value['max_abs_delta'],
                                     'cosine_min': min(value['cosine_similarity_by_row'])}
                              for name, value in bucket_results.items()},
                  'validation_rows_accessed': 0, 'skipped_rows': 0}, ensure_ascii=False, indent=2))


{
  "status": "FAIL_PADDING_CHANGES_REPRESENTATION",
  "artifact": "/content/drive/MyDrive/prelim/data/processed/mission2_raw_waveform_pretrained_speech/stage12_wavlm_d54a03f1f0_871fb4d8c3_3d88f294/engineering_batch64_check_20261008/duration_bucket_padding_20261008T035716Z_263b5217/duration_bucket_padding_equivalence.json",
  "rows": 15,
  "max_abs_delta": 1.6908061504364014,
  "tolerance": 0.002,
  "cosine_similarity_min": 0.143741175532341,
  "buckets": {
    "<0.5": {
      "durations_seconds": [
        0.01,
        0.4,
        0.499
      ],
      "max_abs_delta": 1.6908061504364014,
      "cosine_min": 0.143741175532341
    },
    "0.5-1": {
      "durations_seconds": [
        0.5,
        0.745,
        0.999
      ],
      "max_abs_delta": 0.0718473494052887,
      "cosine_min": 0.9964510202407837
    },
    "1-2": {
      "durations_seconds": [
        1.0,
        1.4,
        1.999
      ],
      "max_abs_delta": 0.09057754278182983,
      "cosine_min": 0.9929122924804688

In [22]:
# CP5 representation-preserving engineering check: run WavLM's GroupNorm feature extractor per waveform, then batch only hidden frames.
import json, uuid
from datetime import datetime, timezone
import numpy as np
import torch
assert model.adapter is None, 'This diagnostic only supports the pinned checkpoint with no adapter.'
assert not model.training and all(not p.requires_grad for p in model.parameters())

def encode_after_individual_feature_extraction(model, waveforms, *, device='cuda', use_amp=True):
    feature_rows, valid_sample_lengths = [], []
    for wave in waveforms:
        padded, valid_lengths, _ = stage12.pad_for_encoder([wave], sample_rate_hz=16000)
        values = torch.from_numpy(padded).to(device=device, dtype=torch.float32)
        with torch.inference_mode():
            if use_amp and str(device).startswith('cuda'):
                with torch.autocast('cuda', dtype=torch.float16):
                    features = model.feature_extractor(values)
            else:
                features = model.feature_extractor(values)
        feature_rows.append(features[0])
        valid_sample_lengths.append(int(valid_lengths[0]))
    max_frames = max(int(row.shape[-1]) for row in feature_rows)
    channels = int(feature_rows[0].shape[0])
    if any(int(row.shape[0]) != channels for row in feature_rows):
        raise ValueError('Feature channel dimension changed across rows')
    padded_features = torch.zeros((len(feature_rows), channels, max_frames), device=device, dtype=feature_rows[0].dtype)
    for row_index, features in enumerate(feature_rows):
        padded_features[row_index, :, :features.shape[-1]] = features
    max_valid_samples = max(valid_sample_lengths)
    sample_positions = torch.arange(max_valid_samples, device=device)[None, :]
    sample_mask = sample_positions < torch.as_tensor(valid_sample_lengths, device=device)[:, None]
    frame_mask = model._get_feature_vector_attention_mask(max_frames, sample_mask.long())
    expected_frames = [int(row.shape[-1]) for row in feature_rows]
    if frame_mask.sum(dim=1).cpu().tolist() != expected_frames:
        raise ValueError(f'Official valid-frame lengths differ from per-wave feature lengths: {frame_mask.sum(dim=1).tolist()} vs {expected_frames}')
    with torch.inference_mode():
        amp_context = torch.autocast('cuda', dtype=torch.float16) if use_amp and str(device).startswith('cuda') else torch.autocast('cpu', enabled=False)
        with amp_context:
            hidden_states, _ = model.feature_projection(padded_features.transpose(1, 2))
            hidden_states = model._mask_hidden_states(hidden_states, attention_mask=frame_mask.long())
            encoded = model.encoder(hidden_states, attention_mask=frame_mask, return_dict=True).last_hidden_state
        weights = frame_mask.unsqueeze(-1).to(dtype=encoded.dtype)
        pooled = ((encoded * weights).sum(dim=1) / weights.sum(dim=1)).float()
        if not torch.isfinite(pooled).all():
            raise ValueError('Nonfinite feature-stage batched embedding')
    return pooled.cpu().numpy().astype(np.float32, copy=False), {'rows': len(waveforms), 'batch_frames': max_frames, 'per_row_feature_frames': expected_frames, 'per_row_valid_frames': frame_mask.sum(dim=1).cpu().tolist()}

FEATURE_TEST_POSITIONS = list(all_positions)
assert len(FEATURE_TEST_POSITIONS) == 15 and not (set(FEATURE_TEST_POSITIONS) & prior_positions)
feature_test_waves = [stage12.crop_and_resample(context, int(pos)) for pos in FEATURE_TEST_POSITIONS]
reference_embeddings = np.vstack([stage12.encode_batch(model, [wave], device='cuda', use_amp=True)[0][0] for wave in feature_test_waves]).astype(np.float32, copy=False)
feature_batch_embeddings, feature_batch_info = encode_after_individual_feature_extraction(model, feature_test_waves, device='cuda', use_amp=True)
feature_deltas = np.max(np.abs(reference_embeddings - feature_batch_embeddings), axis=1)
feature_cosines = np.sum(reference_embeddings * feature_batch_embeddings, axis=1) / (np.linalg.norm(reference_embeddings, axis=1) * np.linalg.norm(feature_batch_embeddings, axis=1))
feature_tolerance = float(PREFLIGHT['batch_size_stability_tolerance'])
feature_status = 'PASS_REPRESENTATION_INVARIANT' if float(feature_deltas.max()) <= feature_tolerance else 'FAIL_REPRESENTATION_DRIFT'
feature_report = {'schema':'mission2_stage12_feature_stage_batching_preflight_v1','status':feature_status,'config_hash':CONFIG_HASH,'data_identity_fingerprint':context.identity['fingerprint'],'source_split':'internal_train_only','rows':len(FEATURE_TEST_POSITIONS),'positions':FEATURE_TEST_POSITIONS,'reference':'unchanged official model forward, one utterance per batch','candidate':'same frozen per-waveform feature extractor; right-pad extracted frame sequences; official hidden mask, projection, encoder and valid-frame pooling','max_abs_delta':float(feature_deltas.max()),'per_row_max_abs_delta':feature_deltas.tolist(),'cosine_similarity_min':float(feature_cosines.min()),'predeclared_tolerance':feature_tolerance,'feature_batch_info':feature_batch_info,'validation_rows_accessed':0,'classifier_or_scaler_fit':False,'skipped_rows':0,'encoder_eval_and_frozen':True}
FEATURE_TEST_ROOT = SENS_ROOT / ('feature_stage_batch_preflight_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
FEATURE_TEST_ROOT.mkdir(parents=True, exist_ok=False)
FEATURE_TEST_PATH = FEATURE_TEST_ROOT / 'feature_stage_batch_preflight.json'
stage12.write_json_new(FEATURE_TEST_PATH, feature_report)
print(json.dumps({'status':feature_status,'artifact':str(FEATURE_TEST_PATH),'rows':len(FEATURE_TEST_POSITIONS),'max_abs_delta':feature_report['max_abs_delta'],'tolerance':feature_tolerance,'cosine_similarity_min':feature_report['cosine_similarity_min'],'feature_batch_info':feature_batch_info,'validation_rows_accessed':0,'skipped_rows':0},ensure_ascii=False,indent=2))


ValueError: Official valid-frame lengths differ from per-wave feature lengths: [1069, 19, 24, 24, 37, 49, 49, 69, 99, 99, 144, 249, 249, 309, 1069] vs [1, 19, 24, 24, 37, 49, 49, 69, 99, 99, 144, 249, 249, 309, 1069]

In [24]:
import inspect, torch, json
print('HF helper source:\n', inspect.getsource(model._get_feature_vector_attention_mask))
print('length helper source:\n', inspect.getsource(model._get_feat_extract_output_lengths))
length_debug=[]
for i, wave in enumerate(feature_test_waves):
    padded, valid, info = stage12.pad_for_encoder([wave], sample_rate_hz=16000)
    length_debug.append({'i':i,'wave':len(wave),'padded_shape':list(padded.shape),'valid':valid.tolist(),'info':info})
print('pad_debug=', json.dumps(length_debug, default=str))
valid_values=torch.as_tensor([int(stage12.pad_for_encoder([w], sample_rate_hz=16000)[1][0]) for w in feature_test_waves],device='cuda')
for width in [400, int(valid_values.max().item()), int(max(len(w) for w in feature_test_waves))]:
    sample_mask=(torch.arange(width,device='cuda')[None,:] < valid_values[:,None]).long()
    output_lengths=model._get_feat_extract_output_lengths(sample_mask.sum(-1))
    print('width',width,'valid',valid_values.tolist(),'formula_out_lengths',output_lengths.tolist())
    print('frame mask counts',model._get_feature_vector_attention_mask(1069,sample_mask).sum(-1).tolist())
# CP5 diagnostic correction: min-receptive-field zero padding is valid for the one output frame.
feature_rows=[]; actual_lengths=[]; effective_lengths=[]
for wave in feature_test_waves:
    padded, valid, _ = stage12.pad_for_encoder([wave], sample_rate_hz=16000)
    values=torch.from_numpy(padded).to(device='cuda', dtype=torch.float32)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        feature_rows.append(model.feature_extractor(values)[0])
    actual_lengths.append(int(valid[0])); effective_lengths.append(max(int(valid[0]), 400))
max_frames=max(int(x.shape[-1]) for x in feature_rows)
channels=int(feature_rows[0].shape[0])
padded_features=torch.zeros((len(feature_rows),channels,max_frames),device='cuda',dtype=feature_rows[0].dtype)
for i,x in enumerate(feature_rows): padded_features[i,:,:x.shape[-1]]=x
mask_width=max(effective_lengths)
sample_mask=(torch.arange(mask_width,device='cuda')[None,:] < torch.as_tensor(effective_lengths,device='cuda')[:,None]).long()
frame_mask=model._get_feature_vector_attention_mask(max_frames,sample_mask)
expected=[int(x.shape[-1]) for x in feature_rows]
assert frame_mask.sum(-1).cpu().tolist()==expected, (frame_mask.sum(-1).tolist(),expected)
with torch.inference_mode(), torch.autocast('cuda',dtype=torch.float16):
    projected,_=model.feature_projection(padded_features.transpose(1,2))
    projected=model._mask_hidden_states(projected,attention_mask=frame_mask.long())
    encoded=model.encoder(projected,attention_mask=frame_mask,return_dict=True).last_hidden_state
weights=frame_mask.unsqueeze(-1).to(encoded.dtype)
custom=((encoded*weights).sum(1)/weights.sum(1)).float().cpu().numpy()
delta=np.max(np.abs(reference_embeddings-custom),axis=1)
cos=np.sum(reference_embeddings*custom,axis=1)/(np.linalg.norm(reference_embeddings,axis=1)*np.linalg.norm(custom,axis=1))
print(json.dumps({'effective_lengths':effective_lengths,'frame_counts':frame_mask.sum(-1).cpu().tolist(),'feature_frames':expected,'max_abs_delta':float(delta.max()),'deltas':delta.tolist(),'cosine_min':float(cos.min()),'tolerance':float(PREFLIGHT['batch_size_stability_tolerance'])},indent=2))


HF helper source:
     def _get_feature_vector_attention_mask(
        self, feature_vector_length: int, attention_mask: torch.LongTensor, add_adapter=None
    ):
        # Effectively attention_mask.sum(-1), but not inplace to be able to run
        # on inference mode.
        non_padded_lengths = attention_mask.cumsum(dim=-1)[:, -1]

        output_lengths = self._get_feat_extract_output_lengths(non_padded_lengths, add_adapter=add_adapter)
        output_lengths = output_lengths.to(torch.long)

        batch_size = attention_mask.shape[0]

        attention_mask = torch.zeros(
            (batch_size, feature_vector_length), dtype=attention_mask.dtype, device=attention_mask.device
        )
        # these two operations makes sure that all values before the output lengths idxs are attended to
        attention_mask[(torch.arange(attention_mask.shape[0], device=attention_mask.device), output_lengths - 1)] = 1
        attention_mask = attention_mask.flip([-1]).cumsum(-1).flip([-1]).b

In [25]:
import inspect
for name, fn in [('model.forward', type(model).forward), ('encoder.forward', type(model.encoder).forward)]:
    lines=inspect.getsource(fn).splitlines()
    print('\n###',name)
    for i,line in enumerate(lines):
        if any(term in line for term in ('attention_mask','feature_projection','_mask_hidden_states','pos_conv_embed','encoder_outputs')):
            print(f'{i+1}: {line}')



### model.forward
7:         attention_mask: torch.Tensor | None = None,
19:         if attention_mask is not None:
20:             # compute reduced attention_mask corresponding to feature vectors
21:             attention_mask = self._get_feature_vector_attention_mask(
22:                 extract_features.shape[1], attention_mask, add_adapter=False
25:         hidden_states, extract_features = self.feature_projection(extract_features)
26:         hidden_states = self._mask_hidden_states(
27:             hidden_states, mask_time_indices=mask_time_indices, attention_mask=attention_mask
30:         encoder_outputs = self.encoder(
32:             attention_mask=attention_mask,
36:         hidden_states = encoder_outputs.last_hidden_state

### encoder.forward
4:         attention_mask: torch.Tensor | None = None,
7:         if attention_mask is not None:
9:             expand_attention_mask = attention_mask.unsqueeze(-1).repeat(1, 1, hidden_states.shape[2])
10:             hidden_states[

In [26]:
def encode_feature_staged(waves):
    rows=[]; eff=[]
    for wave in waves:
        padded,valid,_=stage12.pad_for_encoder([wave],sample_rate_hz=16000)
        x=torch.from_numpy(padded).to(device='cuda',dtype=torch.float32)
        with torch.inference_mode(),torch.autocast('cuda',dtype=torch.float16):
            rows.append(model.feature_extractor(x)[0])
        eff.append(max(int(valid[0]),400))
    t=max(int(r.shape[-1]) for r in rows); c=int(rows[0].shape[0])
    feats=torch.zeros((len(rows),c,t),device='cuda',dtype=rows[0].dtype)
    for i,r in enumerate(rows): feats[i,:,:r.shape[-1]]=r
    sw=max(eff); sm=(torch.arange(sw,device='cuda')[None,:] < torch.as_tensor(eff,device='cuda')[:,None]).long()
    fm=model._get_feature_vector_attention_mask(t,sm)
    with torch.inference_mode(),torch.autocast('cuda',dtype=torch.float16):
        h,_=model.feature_projection(feats.transpose(1,2)); h=model._mask_hidden_states(h,attention_mask=fm.long())
        z=model.encoder(h,attention_mask=fm,return_dict=True).last_hidden_state
    w=fm.unsqueeze(-1).to(z.dtype)
    return ((z*w).sum(1)/w.sum(1)).float().cpu().numpy()

per_bin={}
for name in stage12.DURATION_BINS:
    pos=bucket_results[name]['positions']
    idx=[FEATURE_TEST_POSITIONS.index(p) for p in pos]
    pred=encode_feature_staged([feature_test_waves[i] for i in idx])
    ref=reference_embeddings[idx]
    d=np.max(np.abs(ref-pred),axis=1)
    c=np.sum(ref*pred,axis=1)/(np.linalg.norm(ref,axis=1)*np.linalg.norm(pred,axis=1))
    per_bin[name]={'deltas':d.tolist(),'max_delta':float(d.max()),'cosine_min':float(c.min())}
print(json.dumps({'per_bin':per_bin,'max_delta':max(v['max_delta'] for v in per_bin.values()),'tolerance':float(PREFLIGHT['batch_size_stability_tolerance'])},indent=2))


{
  "per_bin": {
    "<0.5": {
      "deltas": [
        0.008434563875198364,
        0.0004023909568786621,
        0.00426974892616272
      ],
      "max_delta": 0.008434563875198364,
      "cosine_min": 0.9999744296073914
    },
    "0.5-1": {
      "deltas": [
        0.0019585266709327698,
        0.0036915987730026245,
        0.0021670758724212646
      ],
      "max_delta": 0.0036915987730026245,
      "cosine_min": 0.9999772906303406
    },
    "1-2": {
      "deltas": [
        0.001549050211906433,
        0.0025298520922660828,
        0.00029896199703216553
      ],
      "max_delta": 0.0025298520922660828,
      "cosine_min": 0.9999876618385315
    },
    "2-5": {
      "deltas": [
        0.0009306035935878754,
        0.0003674626350402832,
        0.0004353644326329231
      ],
      "max_delta": 0.0009306035935878754,
      "cosine_min": 0.9999988079071045
    },
    ">=5": {
      "deltas": [
        0.00022111833095550537,
        0.0006973147392272949,
        0.

In [27]:
from datetime import datetime, timezone
import json, uuid
FEATURE_TEST_ROOT = SENS_ROOT / ('feature_stage_batch_preflight_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
FEATURE_TEST_ROOT.mkdir(parents=True, exist_ok=False)
feature_report = {
    'schema':'mission2_stage12_feature_stage_batch_preflight_v1',
    'status':'FAIL_REPRESENTATION_EQUIVALENCE',
    'config_hash':CONFIG_HASH, 'data_identity_fingerprint':context.identity['fingerprint'],
    'encoder_repository':CHECKPOINT_INFO['repository'], 'encoder_revision':CHECKPOINT_INFO['revision'],
    'source_split':'internal_train_only', 'positions':[int(x) for x in FEATURE_TEST_POSITIONS],
    'duration_seconds':[float(x) for x in all_durations], 'rows':len(FEATURE_TEST_POSITIONS),
    'reference':'frozen official forward, singleton waveform batches',
    'candidate':'per-waveform feature extractor; pad extracted frame tensors within each existing duration bin; official hidden-frame mask; same AMP and masked mean',
    'minimum_input_length_samples':400, 'valid_length_policy':'max(actual_samples, 400) only for official mask; original sample count retained',
    'all_15_batched_max_abs_delta':float(delta.max()),
    'within_duration_bin':per_bin,
    'within_bin_max_abs_delta':max(v['max_delta'] for v in per_bin.values()),
    'cosine_similarity_min_within_bins':min(v['cosine_min'] for v in per_bin.values()),
    'predeclared_max_abs_tolerance':float(PREFLIGHT['batch_size_stability_tolerance']),
    'raw_waveform_bucket_padding_test':'FAIL; see duration_bucket_padding artifact',
    'validation_rows_accessed':0, 'scaler_or_classifier_fit':False, 'skipped_rows':0,
    'fresh_5000_to_10000_row_benchmark_started':False,
    'reason_benchmark_not_started':'candidate embedding drift exceeds predeclared tolerance; no throughput claim is valid',
    'original_compute_gate_preserved':COMPUTE_GATE['decision'],
    'original_projection_hours':float(COMPUTE_GATE['projected_full_hours']),
    'stage13_started':False
}
FEATURE_TEST_PATH=FEATURE_TEST_ROOT/'feature_stage_batch_preflight.json'
stage12.write_json_new(FEATURE_TEST_PATH, feature_report)
feature_sha=stage12.file_sha256(FEATURE_TEST_PATH)
log_path=REPO_ROOT/'docs'/'mission2_experiment_log.md'
marker='### Stage 12 supplementary duration-bucket batching equivalence check — 2026-10-08'
log_text=log_path.read_text(encoding='utf-8')
if marker not in log_text:
    section=f'''

{marker}

**Purpose:** validate a representation-preserving duration-bucket throughput path before any fresh benchmark. The original Stage 12 compute gate remains {COMPUTE_GATE['decision']} at {COMPUTE_GATE['projected_full_hours']:.4f} h.

- **Data:** 15 fresh internal-training rows across the five duration bins; no Validation access, no classifier/scaler fit, and zero skipped rows.
- **Raw-waveform right-padding:** failed representation equivalence within the five duration bins; maximum absolute embedding delta 1.690806 against the predeclared 0.002 tolerance. Artifact: {BUCKET_PADDING_PATH.as_posix()}.
- **Representation-preserving candidate:** ran the frozen WavLM feature extractor per waveform to preserve GroupNorm statistics, then padded only extracted frame sequences within each duration bin and used the official hidden-frame mask. Maximum within-bin absolute delta was {feature_report['within_bin_max_abs_delta']:.8f}; all-15 padded batch maximum delta was {feature_report['all_15_batched_max_abs_delta']:.8f}. Both exceed 0.002, so this candidate failed preflight. Artifact: {FEATURE_TEST_PATH.as_posix()} (SHA-256 {feature_sha}).
- **Decision:** no new 5,000–10,000-row throughput benchmark was started, and no new runtime projection or compute-gate decision is claimed. The earlier batch-64 same-row result remains supplementary and does not replace CP5. Full embeddings/classifier evaluation remain unstarted; Stage 13 not started.
'''
    with log_path.open('a',encoding='utf-8',newline='') as f: f.write(section)
log_text=log_path.read_text(encoding='utf-8')
assert log_text.count(marker)==1 and FEATURE_TEST_PATH.exists()
print(json.dumps({'status':feature_report['status'],'artifact':str(FEATURE_TEST_PATH),'sha256':feature_sha,'within_bin_max_abs_delta':feature_report['within_bin_max_abs_delta'],'combined_max_abs_delta':feature_report['all_15_batched_max_abs_delta'],'tolerance':feature_report['predeclared_max_abs_tolerance'],'fresh_benchmark_started':feature_report['fresh_5000_to_10000_row_benchmark_started'],'log_marker_count':log_text.count(marker),'original_gate':COMPUTE_GATE['decision'],'stage13_started':False},indent=2))

{
  "status": "FAIL_REPRESENTATION_EQUIVALENCE",
  "artifact": "/content/drive/MyDrive/prelim/data/processed/mission2_raw_waveform_pretrained_speech/stage12_wavlm_d54a03f1f0_871fb4d8c3_3d88f294/engineering_batch64_check_20261008/feature_stage_batch_preflight_20261008T040524Z_c4456a69/feature_stage_batch_preflight.json",
  "sha256": "14ef77f96f7c5a69d340f9721cec2392117b17174e218776d0235f7a0f7fbb16",
  "within_bin_max_abs_delta": 0.008434563875198364,
  "combined_max_abs_delta": 0.019711598753929138,
  "tolerance": 0.002,
  "fresh_benchmark_started": false,
  "log_marker_count": 1,
  "original_gate": "RESOURCE/TIME-BUDGET INFEASIBLE",
  "stage13_started": false
}


In [28]:
equal_frame_groups=[[2,3],[5,6],[8,9],[11,12]]
equal_frame_results=[]
for idx in equal_frame_groups:
    pred=encode_feature_staged([feature_test_waves[i] for i in idx])
    ref=reference_embeddings[idx]
    d=np.max(np.abs(ref-pred),axis=1)
    c=np.sum(ref*pred,axis=1)/(np.linalg.norm(ref,axis=1)*np.linalg.norm(pred,axis=1))
    equal_frame_results.append({'indices':idx,'positions':[FEATURE_TEST_POSITIONS[i] for i in idx], 'frame_lengths':[feature_batch_info['per_row_valid_frames'][0] if False else int(model._get_feat_extract_output_lengths(torch.tensor(max(len(feature_test_waves[i]),400),device='cuda')).item()) for i in idx], 'deltas':d.tolist(),'max_delta':float(d.max()),'cosine_min':float(c.min())})
print(json.dumps({'groups':equal_frame_results,'max_delta':max(x['max_delta'] for x in equal_frame_results),'tolerance':float(PREFLIGHT['batch_size_stability_tolerance'])},indent=2))


{
  "groups": [
    {
      "indices": [
        2,
        3
      ],
      "positions": [
        881245,
        60
      ],
      "frame_lengths": [
        24,
        24
      ],
      "deltas": [
        0.0004938691854476929,
        0.002183869481086731
      ],
      "max_delta": 0.002183869481086731,
      "cosine_min": 0.9999920725822449
    },
    {
      "indices": [
        5,
        6
      ],
      "positions": [
        887643,
        222
      ],
      "frame_lengths": [
        49,
        49
      ],
      "deltas": [
        0.001006200909614563,
        0.0023982152342796326
      ],
      "max_delta": 0.0023982152342796326,
      "cosine_min": 0.9999921321868896
    },
    {
      "indices": [
        8,
        9
      ],
      "positions": [
        881153,
        889
      ],
      "frame_lengths": [
        99,
        99
      ],
      "deltas": [
        0.0010235905647277832,
        0.000969231128692627
      ],
      "max_delta": 0.001023590564727783

In [29]:
import json, uuid
from datetime import datetime, timezone
SAME_FRAME_ROOT=SENS_ROOT/('same_frame_batch_preflight_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8])
SAME_FRAME_ROOT.mkdir(parents=True,exist_ok=False)
same_frame_max=max(x['max_delta'] for x in equal_frame_results)
same_frame_report={
 'schema':'mission2_stage12_same_frame_batch_preflight_v1','status':'FAIL_REPRESENTATION_EQUIVALENCE' if same_frame_max>float(PREFLIGHT['batch_size_stability_tolerance']) else 'PASS_REPRESENTATION_EQUIVALENCE',
 'config_hash':CONFIG_HASH,'data_identity_fingerprint':context.identity['fingerprint'],'encoder_repository':CHECKPOINT_INFO['repository'],'encoder_revision':CHECKPOINT_INFO['revision'],
 'source_split':'internal_train_only','reference':'frozen official model forward, singleton waveform batches','candidate':'per-waveform frozen feature extractor; batch transformer only for rows with identical valid hidden-frame count; no padded hidden frames',
 'groups':equal_frame_results,'rows':sum(len(x['indices']) for x in equal_frame_results),'max_abs_delta':same_frame_max,'predeclared_tolerance':float(PREFLIGHT['batch_size_stability_tolerance']),
 'validation_rows_accessed':0,'classifier_or_scaler_fit':False,'skipped_rows':0,'fresh_benchmark_started':False,'original_compute_gate':COMPUTE_GATE['decision'],'original_projected_hours':float(COMPUTE_GATE['projected_full_hours']),'stage13_started':False}
SAME_FRAME_PATH=SAME_FRAME_ROOT/'same_frame_batch_preflight.json'
stage12.write_json_new(SAME_FRAME_PATH,same_frame_report)
same_frame_sha=stage12.file_sha256(SAME_FRAME_PATH)
log_path=REPO_ROOT/'docs'/'mission2_experiment_log.md'
marker2='### Stage 12 supplementary equal-frame batching equivalence check — 2026-10-08'
text2=log_path.read_text(encoding='utf-8')
if marker2 not in text2:
    section2=f'''

{marker2}

**Purpose:** one last representation-preserving microbatch check after raw-waveform and variable hidden-frame padding failed. The frozen feature extractor ran per utterance; only the Transformer encoder was batched for equal valid-frame lengths, with no hidden-frame padding.

- **Sample:** 8 internal-training rows drawn from the earlier 15-row equivalence diagnostic (4 pairs with equal valid-frame counts 24, 49, 99, and 249); no Validation access, fit, or skipped row.
- **Result:** maximum absolute embedding delta {same_frame_max:.8f} against the predeclared {same_frame_report['predeclared_tolerance']:.3f} tolerance; the check failed. The largest pair deltas were {', '.join(f"{x['max_delta']:.8f}" for x in equal_frame_results)}.
- **Decision:** no fresh throughput benchmark or new compute projection was run. Original CP5 remains {COMPUTE_GATE['decision']} at {COMPUTE_GATE['projected_full_hours']:.4f} h; full extraction/classifier evaluation remain unstarted; Stage 13 not started.
- **Artifact:** {SAME_FRAME_PATH.as_posix()} (SHA-256 {same_frame_sha}).
'''
    with log_path.open('a',encoding='utf-8',newline='') as f:f.write(section2)
text2=log_path.read_text(encoding='utf-8')
assert text2.count(marker2)==1 and SAME_FRAME_PATH.exists()
print(json.dumps({'status':same_frame_report['status'],'artifact':str(SAME_FRAME_PATH),'sha256':same_frame_sha,'max_delta':same_frame_max,'tolerance':same_frame_report['predeclared_tolerance'],'original_gate':COMPUTE_GATE['decision'],'log_marker_count':text2.count(marker2),'stage13_started':False},indent=2))

{
  "status": "FAIL_REPRESENTATION_EQUIVALENCE",
  "artifact": "/content/drive/MyDrive/prelim/data/processed/mission2_raw_waveform_pretrained_speech/stage12_wavlm_d54a03f1f0_871fb4d8c3_3d88f294/engineering_batch64_check_20261008/same_frame_batch_preflight_20261008T040752Z_3ebbc60a/same_frame_batch_preflight.json",
  "sha256": "114ec217e99fce2f4d9e710038001668fe3b4e700c8ef085739aff529a088193",
  "max_delta": 0.0023982152342796326,
  "tolerance": 0.002,
  "original_gate": "RESOURCE/TIME-BUDGET INFEASIBLE",
  "log_marker_count": 1,
  "stage13_started": false
}


In [32]:
# CP5b-1 — Reconfirm existing gate and compare two deterministic padding rules from Training lengths only.
import json
from pathlib import Path
import numpy as np
assert 'context' in globals() and 'stage12' in globals() and 'model' in globals()
assert context.identity['fingerprint']=='871fb4d8c323579a4398adc4119c8ec3583b218545183b7578d40ef9c16e4b82'
assert len(context.train_idx)==711467 and len(context.validation_idx)==177891
old_root=Path(RUN_ROOT); old_benchmark=stage12.read_json(old_root/'throughput_benchmark.json'); old_gate=stage12.read_json(old_root/'compute_gate.json')
legacy_paths={'warm64':old_root/'engineering_batch64_check_20261008'/'batch_size_sensitivity.json','raw_pad':old_root/'engineering_batch64_check_20261008'/'duration_bucket_padding_20261008T035716Z_263b5217'/'duration_bucket_padding_equivalence.json','feature_pad':old_root/'engineering_batch64_check_20261008'/'feature_stage_batch_preflight_20261008T040524Z_c4456a69'/'feature_stage_batch_preflight.json','same_frame':old_root/'engineering_batch64_check_20261008'/'same_frame_batch_preflight_20261008T040752Z_3ebbc60a'/'same_frame_batch_preflight.json'}
legacy={k:stage12.read_json(v) for k,v in legacy_paths.items()}
assert old_benchmark['benchmark_utterances']==3000 and abs(old_gate['projected_full_hours']-45.369999432472035)<1e-6
print('warm64 keys:',list(legacy['warm64']))
print(json.dumps({'warm64':legacy['warm64'],'raw_pad':legacy['raw_pad'],'feature_pad':legacy['feature_pad'],'same_frame':legacy['same_frame']},indent=2)[:7000])
train=context.source.iloc[np.asarray(context.train_idx,dtype=np.int64)]
lengths=2*(train.crop_end_frame.to_numpy(dtype=np.int64)-train.crop_start_frame.to_numpy(dtype=np.int64)); assert len(lengths)==711467 and (lengths>0).all()
def pow2_target(n): return max(400,1<<(int(n)-1).bit_length())
def quarter_second_target(n): return max(400,((int(n)+3999)//4000)*4000)
def summarize_rule(name,fn):
 targets=np.fromiter((fn(int(n)) for n in lengths),dtype=np.int64,count=len(lengths)); valid=np.maximum(lengths,400); counts=np.unique(targets,return_counts=True)[1]
 def batch_summary(b):
  sizes=[]
  for c in counts:
   q,r=divmod(int(c),b); sizes.extend([b]*q); sizes.extend([r] if r else [])
  sizes=np.asarray(sizes,dtype=np.int64)
  return {'mean_rows_per_forward':float(sizes.mean()),'full_batch_fraction':float((sizes==b).mean()),'forward_count':int(len(sizes)),'min_rows':int(sizes.min()),'p50_rows':float(np.median(sizes))}
 return {'name':name,'unique_target_lengths':int(len(counts)),'padding_ratio':float(targets.sum()/valid.sum()),'padding_waste_fraction':float((targets-valid).sum()/targets.sum()),'short_rows':int((lengths<400).sum()),'group_count_min':int(counts.min()),'group_count_p50':float(np.median(counts)),'group_count_max':int(counts.max()),'batch32':batch_summary(32),'batch64':batch_summary(64)}
RULE_SUMMARIES=[summarize_rule('next_power_of_two_samples',pow2_target),summarize_rule('250ms_multiple_4000_samples',quarter_second_target)]
print(json.dumps({'prior':{'old_rows_per_sec':old_benchmark['rows_per_second'],'old_gate':old_gate['decision'],'old_hours':old_gate['projected_full_hours'],'warm64_rows_per_sec':legacy['warm64'].get('rows_per_second'),'raw_padding_max_abs':legacy['raw_pad'].get('max_abs_delta'),'feature_within_bin_max_abs':legacy['feature_pad'].get('within_bin_max_abs_delta'),'same_frame_max_abs':legacy['same_frame'].get('max_abs_delta')},'training_length_samples16k':{'min':int(lengths.min()),'p50':float(np.median(lengths)),'p95':float(np.quantile(lengths,.95)),'max':int(lengths.max())},'rules':RULE_SUMMARIES,'validation_rows_read':0},indent=2))

warm64 keys: ['baseline_batch8', 'batch64', 'colab_units_per_hour_displayed_before', 'colab_usage_units_available_before', 'config_hash', 'data_identity_fingerprint', 'full_extraction_started', 'original_compute_gate_preserved', 'projection_ratio_vs_batch8', 'purpose', 'run_root', 'same_rows_as_batch8', 'sample_position_sha256', 'sample_rows', 'schema', 'sensitivity_projection_gate_status', 'stage13_started', 'throughput_ratio_vs_batch8']
{
  "warm64": {
    "baseline_batch8": {
      "audio_seconds_per_second": 10.801945015750894,
      "projected_full_hours": 45.369999432472035,
      "rows_per_second": 5.4480280340098615
    },
    "batch64": {
      "amp": true,
      "artifact_budget_seconds": 63.85153330297171,
      "audio_seconds_per_second": 175.46937486810705,
      "batch_size_max_observed": 10,
      "batch_size_min_observed": 1,
      "batch_size_requested": 64,
      "benchmark_artifact_dir": "/content/drive/MyDrive/prelim/data/processed/mission2_raw_waveform_pretrained_s

In [34]:
# CP5b-2 — Frozen-candidate representation stability on fresh internal-Training rows only.
# Acceptance was fixed before this encoder call: cosine min>=0.999, median>=0.9999;
# relative-L2 median<=0.005, p99<=0.01; finite values; coherent mean shift ratio<=0.001.
import hashlib, json, uuid
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch
assert CHECKPOINT_INFO['repository']=='microsoft/wavlm-base-plus'
assert CHECKPOINT_INFO['revision']=='4c66d4806a428f2e922ccfa1a962776e232d487b'
assert not model.training and all(not p.requires_grad for p in model.parameters())
# Reserve prior 3,000 benchmark positions plus every old engineering-diagnostic position.
prior_positions=set()
for info in old_benchmark['duration_bins'].values(): prior_positions.update(map(int,info['sample_positions']))
assert len(prior_positions)==3000
for field in ('positions',): prior_positions.update(map(int,legacy['raw_pad'].get(field,[])))
prior_positions.update(map(int,legacy['feature_pad'].get('positions',[])))
for group in legacy['same_frame'].get('groups',[]): prior_positions.update(map(int,group.get('positions',[])))
train_pos=np.asarray(context.train_idx,dtype=np.int64)
train_src=context.source.iloc[train_pos]
train_dur=(train_src.crop_end_frame.to_numpy(dtype=np.int64)-train_src.crop_start_frame.to_numpy(dtype=np.int64))/8000.0
train_len16k=2*(train_src.crop_end_frame.to_numpy(dtype=np.int64)-train_src.crop_start_frame.to_numpy(dtype=np.int64))
quarter_targets=np.fromiter((quarter_second_target(int(n)) for n in train_len16k),dtype=np.int64,count=len(train_len16k))
bin_masks={'<0.5':train_dur<.5,'0.5-1':(train_dur>=.5)&(train_dur<1),'1-2':(train_dur>=1)&(train_dur<2),'2-5':(train_dur>=2)&(train_dur<5),'>=5':train_dur>=5}
rng=np.random.default_rng(42)
COHORT_POSITIONS={}; COHORT_TARGETS={}; cohort_counts={}
for name,mask in bin_masks.items():
    target_values,counts=np.unique(quarter_targets[mask],return_counts=True)
    candidates=sorted(zip(counts,target_values),reverse=True)
    chosen=None
    for _,target in candidates:
        locals_=np.flatnonzero(mask&(quarter_targets==target))
        globals_=train_pos[locals_]
        globals_=globals_[~np.isin(globals_,np.fromiter(prior_positions,dtype=np.int64))]
        need=64 if name=='1-2' else 32
        if len(globals_)>=need:
            chosen=(int(target),globals_); break
    if chosen is None: raise RuntimeError(f'Could not select fresh rows for {name}; no rows are dropped or substituted.')
    target,available=chosen; need=64 if name=='1-2' else 32
    selected=np.sort(rng.choice(available,size=need,replace=False).astype(np.int64))
    COHORT_POSITIONS[name]=selected; COHORT_TARGETS[name]=target; cohort_counts[name]=len(available)
STABILITY_POSITIONS=np.concatenate([COHORT_POSITIONS[name] for name in ('<0.5','0.5-1','1-2','2-5','>=5')])
assert len(STABILITY_POSITIONS)==192 and len(set(map(int,STABILITY_POSITIONS)))==192
assert not (set(map(int,STABILITY_POSITIONS)) & prior_positions)
# Freeze the 2 candidate rules and numerical gates before model results are observed.
STABILITY_RULES={'next_power_of_two_samples':pow2_target,'250ms_multiple_4000_samples':quarter_second_target}
STABILITY_THRESHOLDS={'cosine_minimum':0.999,'cosine_median':0.9999,'relative_l2_median':0.005,'relative_l2_p99':0.01,'coherent_mean_shift_ratio_max':0.001,'nonfinite_allowed':0,'skipped_rows_allowed':0}
_previous_stability_config=globals().get('STABILITY_CONFIG')
_previous_engineering_root=globals().get('ENGINEERING_ROOT')
ENGINEERING_ROOT=Path(OUTPUT_PARENT)/('stage12_wavlm_canonical_reassessment_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8])
ENGINEERING_ROOT.mkdir(parents=True,exist_ok=False)
STABILITY_CONFIG={'schema':'mission2_stage12_canonical_engineering_preflight_v1','prior_config_hash':CONFIG_HASH,'data_identity_fingerprint':context.identity['fingerprint'],'model_repository':CHECKPOINT_INFO['repository'],'model_revision':CHECKPOINT_INFO['revision'],'source_split':'internal_train_only','rules':{'next_power_of_two_samples':'max(400, 2**ceil(log2(n_samples)))','250ms_multiple_4000_samples':'max(400, ceil(n_samples/4000)*4000)'},'sample_plan':{'rows':192,'duration_bins':list(COHORT_POSITIONS),'rows_by_bin':{k:len(v) for k,v in COHORT_POSITIONS.items()},'fresh_positions_sha256':hashlib.sha256(STABILITY_POSITIONS.astype('<i8').tobytes()).hexdigest(),'excluded_prior_benchmark_or_diagnostic_rows':len(prior_positions)},'comparisons':['singleton repeat','batch sizes 4/8/16/32','batch size 64 where available','sample order permutation','changed companion grouping','repeat identical batch'],'thresholds':STABILITY_THRESHOLDS,'amp':'float16 autocast on CUDA','validation_rows_read':0,'accuracy_or_loss_used':False}
STABILITY_CONFIG['previous_attempt']={'artifact_root':str(_previous_engineering_root) if _previous_engineering_root else None,'failure':'Selected cohort had mixed next-power-of-two targets; corrected to group forward batches by each frozen rule target.'}
stage12.write_json_new(ENGINEERING_ROOT/'stability_config.json',STABILITY_CONFIG)
# Save exact row/key/call/label and duration tracking; labels remain outside all encoder inputs.
sample_rows=stage12._row_records(context,STABILITY_POSITIONS)
sample_frame=context.source.iloc[STABILITY_POSITIONS]
sample_identity=[{'row_position':int(pos),'record_key':key,'call_id':call,'y_true':int(label),'duration_seconds':float((int(row.crop_end_frame)-int(row.crop_start_frame))/8000.0),'duration_bin':stage12.duration_bin((int(row.crop_end_frame)-int(row.crop_start_frame))/8000.0)} for pos,key,call,label,(_,row) in zip(STABILITY_POSITIONS,sample_rows['record_key'],sample_rows['call_id'],sample_rows['y_true'],sample_frame.iterrows())]
assert len({x['record_key'] for x in sample_identity})==192 and len({x['call_id'] for x in sample_identity})>0
stage12.write_json_new(ENGINEERING_ROOT/'stability_sample_identity.json',{'rows':sample_identity,'source_split':'internal_train_only','labels_used_as_model_input':False})
# Reuse only the exact in-memory crop cache from the failed attempt in this same live runtime.
_position_sha=hashlib.sha256(STABILITY_POSITIONS.astype('<i8').tobytes()).hexdigest()
_previous_waves=globals().get('waves_by_pos')
_reuse_previous_waves=(isinstance(_previous_stability_config,dict) and _previous_stability_config.get('data_identity_fingerprint')==context.identity['fingerprint'] and _previous_stability_config.get('model_repository')==CHECKPOINT_INFO['repository'] and _previous_stability_config.get('model_revision')==CHECKPOINT_INFO['revision'] and _previous_stability_config.get('sample_plan',{}).get('fresh_positions_sha256')==_position_sha and isinstance(_previous_waves,dict) and set(_previous_waves)==set(map(int,STABILITY_POSITIONS)))
if _reuse_previous_waves:
    waves_by_pos=_previous_waves
else:
    # WAV load/crop/resample uses the existing organizer ZIP mapping and manifest crop.
    archive_reader=getattr(context,'archive_reader',None)
    if archive_reader is not None and hasattr(archive_reader,'_wave_cache'):
        archive_reader._wave_cache.clear(); archive_reader.cache_wavs=0
    waves_by_pos={int(pos):stage12.crop_and_resample(context,int(pos)) for pos in STABILITY_POSITIONS}
for pos,wave in waves_by_pos.items():
    expected=int(train_len16k[np.flatnonzero(train_pos==pos)[0]])
    if len(wave)!=expected or len(wave)==0 or not np.isfinite(wave).all(): raise ValueError(f'WAV crop/resample length or finite check failed for row {pos}')

def canonical_encode_positions(positions,rule_fn):
    positions=[int(x) for x in positions]
    batch_waves=[waves_by_pos[p] for p in positions]
    lengths=np.asarray([len(w) for w in batch_waves],dtype=np.int64)
    targets=np.asarray([rule_fn(int(n)) for n in lengths],dtype=np.int64)
    if len(set(targets.tolist()))!=1: raise ValueError('A forward batch must have one deterministic target length.')
    target=int(targets[0])
    if target<400 or int(lengths.max())>target: raise ValueError('Canonical pad target truncates or violates the encoder minimum.')
    padded=np.zeros((len(positions),target),dtype=np.float32)
    for i,w in enumerate(batch_waves): padded[i,:len(w)]=w
    values=torch.from_numpy(padded).to(device='cuda',dtype=torch.float32)
    attention=(torch.arange(target,device='cuda')[None,:] < torch.as_tensor(lengths,device='cuda')[:,None]).long()
    with torch.inference_mode():
        with torch.autocast('cuda',dtype=torch.float16):
            output=model(input_values=values,attention_mask=attention,return_dict=True)
            hidden=output.last_hidden_state
            frame_mask=stage12._model_hidden_mask(model,attention,hidden.shape[1])
            weight=frame_mask.unsqueeze(-1).to(hidden.dtype)
            denom=weight.sum(dim=1)
            if (denom<=0).any(): raise ValueError('WavLM returned zero valid hidden frames.')
            pooled=((hidden*weight).sum(dim=1)/denom).float()
    result=pooled.cpu().numpy().astype(np.float32,copy=False)
    if result.shape!=(len(positions),768) or not np.isfinite(result).all(): raise ValueError('Canonical encoder output shape/finite check failed.')
    return result

def compare_pair(reference,candidate):
    ref=np.asarray(reference,dtype=np.float64); cur=np.asarray(candidate,dtype=np.float64)
    diff=cur-ref; refnorm=np.linalg.norm(ref,axis=1); curnorm=np.linalg.norm(cur,axis=1)
    cos=(ref*cur).sum(axis=1)/np.maximum(refnorm*curnorm,1e-30)
    rel=np.linalg.norm(diff,axis=1)/np.maximum(refnorm,1e-30)
    return {'cosine':cos,'relative_l2':rel,'mean_abs':np.mean(np.abs(diff),axis=1),'max_abs':np.max(np.abs(diff),axis=1),'signed_shift_ratio':float(np.linalg.norm(diff.mean(axis=0))/max(float(refnorm.mean()),1e-30))}

def run_condition(rule_name,condition,positions,embeddings,baseline,condition_rows):
    metrics=compare_pair(np.vstack([baseline[int(p)] for p in positions]),embeddings)
    condition_rows.setdefault(condition,[]).append(metrics)

# Re-run the legacy unpadded singleton once only as a descriptive comparator, not a rejection gate.
legacy_singletons={int(pos):stage12.encode_batch(model,[waves_by_pos[int(pos)]],device='cuda',use_amp=True)[0][0] for pos in STABILITY_POSITIONS}
STABILITY_RESULTS={}
for rule_name,rule_fn in STABILITY_RULES.items():
    singleton={int(pos):canonical_encode_positions([int(pos)],rule_fn)[0] for pos in STABILITY_POSITIONS}
    conditions={}
    repeated=np.vstack([canonical_encode_positions([int(pos)],rule_fn)[0] for pos in STABILITY_POSITIONS])
    run_condition(rule_name,'singleton_repeat',STABILITY_POSITIONS,repeated,singleton,conditions)
        # Keep each forward batch within one deterministic target length for the rule under test.
    tested_batch_plan={}
    def encode_grouped(positions,rule_fn,batch_size):
        positions=[int(x) for x in positions]
        groups_by_target={}
        for pos in positions: groups_by_target.setdefault(int(rule_fn(len(waves_by_pos[pos]))),[]).append(pos)
        encoded={}; actual_sizes=[]
        for target,group_positions in sorted(groups_by_target.items()):
            for offset in range(0,len(group_positions),batch_size):
                part=group_positions[offset:offset+batch_size]
                batch_embeddings=canonical_encode_positions(part,rule_fn)
                encoded.update({int(pos):embedding for pos,embedding in zip(part,batch_embeddings)})
                actual_sizes.append(len(part))
        if set(encoded)!=set(positions): raise ValueError('Grouped stability forward omitted or duplicated rows.')
        return np.vstack([encoded[pos] for pos in positions]), {str(t):len(v) for t,v in sorted(groups_by_target.items())}, sorted(set(actual_sizes))
    for bin_name,all_group in COHORT_POSITIONS.items():
        positions=[int(x) for x in all_group]
        target_groups={}
        for pos in positions: target_groups.setdefault(int(rule_fn(len(waves_by_pos[pos]))),[]).append(pos)
        tested_batch_plan[bin_name]={'rows':len(positions),'pad_target_group_sizes':{str(t):len(v) for t,v in sorted(target_groups.items())},'batch_sizes':{}}
        for batch_size in (4,8,16,32):
            out,group_sizes,actual_sizes=encode_grouped(positions,rule_fn,batch_size)
            run_condition(rule_name,f'{bin_name}:batch{batch_size}',positions,out,singleton,conditions)
            tested_batch_plan[bin_name]['batch_sizes'][str(batch_size)]={'rows_compared':len(positions),'group_sizes':group_sizes,'actual_forward_batch_sizes':actual_sizes}
        # Permute companions only within equal target-length groups, preserving deterministic padding.
        permuted=[]
        for target,group_positions in sorted(target_groups.items()): permuted.extend(rng.permutation(group_positions).tolist())
        perm_out,perm_groups,perm_sizes=encode_grouped(permuted,rule_fn,32)
        run_condition(rule_name,f'{bin_name}:batch32_permuted',permuted,perm_out,singleton,conditions)
        rotated=[]
        for target,group_positions in sorted(target_groups.items()): rotated.extend(np.roll(group_positions,5).tolist())
        rot_out,rot_groups,rot_sizes=encode_grouped(rotated,rule_fn,8)
        run_condition(rule_name,f'{bin_name}:batch8_companion_rotation',rotated,rot_out,singleton,conditions)
        repeated_out,repeat_groups,repeat_sizes=encode_grouped(positions,rule_fn,32)
        run_condition(rule_name,f'{bin_name}:batch32_repeat',positions,repeated_out,singleton,conditions)
        tested_batch_plan[bin_name]['permutation']={'rows_compared':len(permuted),'group_sizes':perm_groups,'actual_forward_batch_sizes':perm_sizes}
        tested_batch_plan[bin_name]['companion_rotation']={'rows_compared':len(rotated),'group_sizes':rot_groups,'actual_forward_batch_sizes':rot_sizes}
        tested_batch_plan[bin_name]['repeat_identical_batch']={'rows_compared':len(positions),'group_sizes':repeat_groups,'actual_forward_batch_sizes':repeat_sizes}
        if bin_name=='1-2':
            available64=[(target,group_positions) for target,group_positions in sorted(target_groups.items()) if len(group_positions)>=64]
            tested_batch_plan[bin_name]['batch64_available']=bool(available64)
            if available64:
                target,group_positions=available64[0]
                out64=canonical_encode_positions(group_positions[:64],rule_fn)
                run_condition(rule_name,'1-2:batch64',group_positions[:64],out64,singleton,conditions)
                repeat64=canonical_encode_positions(group_positions[:64],rule_fn)
                run_condition(rule_name,'1-2:batch64_repeat',group_positions[:64],repeat64,singleton,conditions)
                tested_batch_plan[bin_name]['batch64']={'rows_compared':64,'pad_target':int(target),'actual_forward_batch_sizes':[64,64]}
            else:
                tested_batch_plan[bin_name]['batch64']={'rows_compared':0,'reason':'No single deterministic target-length group contains 64 sampled rows.'}
    flattened={key:np.concatenate([m[key] for condition_metrics in conditions.values() for m in condition_metrics]) for key in ('cosine','relative_l2','mean_abs','max_abs')}

    per_condition={}
    for name,groups in conditions.items():
        vals={key:np.concatenate([m[key] for m in groups]) for key in ('cosine','relative_l2','mean_abs','max_abs')}
        shifts=[m['signed_shift_ratio'] for m in groups]
        per_condition[name]={'comparisons':int(len(vals['cosine'])),'cosine_min':float(vals['cosine'].min()),'cosine_median':float(np.median(vals['cosine'])),'relative_l2_median':float(np.median(vals['relative_l2'])),'relative_l2_p99':float(np.quantile(vals['relative_l2'],.99)),'mean_abs_mean':float(vals['mean_abs'].mean()),'max_abs_max':float(vals['max_abs'].max()),'coherent_mean_shift_ratio_max':float(max(shifts)),'finite':bool(all(np.isfinite(v).all() for v in vals.values()))}
    old_diffs=[compare_pair(legacy_singletons[int(pos)][None,:],singleton[int(pos)][None,:]) for pos in STABILITY_POSITIONS]
    old_summary={'rows':len(old_diffs),'cosine_min':float(min(x['cosine'][0] for x in old_diffs)),'cosine_median':float(np.median([x['cosine'][0] for x in old_diffs])),'relative_l2_median':float(np.median([x['relative_l2'][0] for x in old_diffs])),'relative_l2_p99':float(np.quantile([x['relative_l2'][0] for x in old_diffs],.99)),'mean_abs_mean':float(np.mean([x['mean_abs'][0] for x in old_diffs])),'max_abs_max':float(np.max([x['max_abs'][0] for x in old_diffs])),'acceptance_role':'descriptive only; old unpadded representation is not the new canonical stability baseline'}
    passed=(all(x['finite'] and x['cosine_min']>=STABILITY_THRESHOLDS['cosine_minimum'] and x['cosine_median']>=STABILITY_THRESHOLDS['cosine_median'] and x['relative_l2_median']<=STABILITY_THRESHOLDS['relative_l2_median'] and x['relative_l2_p99']<=STABILITY_THRESHOLDS['relative_l2_p99'] and x['coherent_mean_shift_ratio_max']<=STABILITY_THRESHOLDS['coherent_mean_shift_ratio_max'] for x in per_condition.values()))
    STABILITY_RESULTS[rule_name]={'status':'PASS' if passed else 'FAIL','rows':192,'cohort_available_rows':cohort_counts,'cohort_pad_targets_250ms':COHORT_TARGETS,'tested_batch_plan':tested_batch_plan,'conditions':per_condition,'aggregate':{'cosine_min':float(flattened['cosine'].min()),'cosine_median':float(np.median(flattened['cosine'])),'relative_l2_median':float(np.median(flattened['relative_l2'])),'relative_l2_p99':float(np.quantile(flattened['relative_l2'],.99)),'mean_abs_mean':float(flattened['mean_abs'].mean()),'mean_abs_p99':float(np.quantile(flattened['mean_abs'],.99)),'max_abs_max':float(flattened['max_abs'].max()),'coherent_mean_shift_ratio_max':float(max(v['coherent_mean_shift_ratio_max'] for v in per_condition.values()))},'old_unpadded_comparison':old_summary,'status_rule':'every condition satisfies all frozen numerical thresholds; per-condition values are retained to reveal any batch-size or order trend'}
# No silent drops: each tested row has a singleton and every requested condition returned an embedding.
assert all(result['rows']==192 for result in STABILITY_RESULTS.values())
assert len(sample_rows['record_key'])==192 and len(set(sample_rows['record_key']))==192
report={'schema':'mission2_stage12_canonical_batch_stability_v1','created_utc':datetime.now(timezone.utc).isoformat(),'prior_gate_preserved':{'decision':old_gate['decision'],'projected_hours':old_gate['projected_full_hours']},'config_hash':CONFIG_HASH,'data_identity_fingerprint':context.identity['fingerprint'],'model_repository':CHECKPOINT_INFO['repository'],'model_revision':CHECKPOINT_INFO['revision'],'source_split':'internal_train_only','sample_rows':192,'sample_positions_sha256':STABILITY_CONFIG['sample_plan']['fresh_positions_sha256'],'duration_bins':{k:{'rows':len(v),'quarter_second_pad_target':COHORT_TARGETS[k],'available_fresh_rows_at_target':cohort_counts[k]} for k,v in COHORT_POSITIONS.items()},'candidate_results':STABILITY_RESULTS,'skipped_rows':0,'validation_rows_read':0,'classifier_or_scaler_fit':False,'official_validation_used':False,'stage13_started':False}
stage12.write_json_new(ENGINEERING_ROOT/'batch_stability_report.json',report)
stage12.write_json_new(ENGINEERING_ROOT/'candidate_comparison.json',{'training_distribution_rule_comparison':RULE_SUMMARIES,'stability_thresholds':STABILITY_THRESHOLDS,'sample_plan':STABILITY_CONFIG['sample_plan'],'results':{k:{'status':v['status'],'aggregate':v['aggregate'],'old_unpadded_comparison':v['old_unpadded_comparison']} for k,v in STABILITY_RESULTS.items()},'validation_rows_read':0,'accuracy_or_loss_used':False})
print(json.dumps({'artifact_root':str(ENGINEERING_ROOT),'fresh_rows':192,'prior_rows_excluded':len(prior_positions),'thresholds':STABILITY_THRESHOLDS,'candidate_stability':{k:{'status':v['status'],'aggregate':v['aggregate'],'old_unpadded_comparison':v['old_unpadded_comparison']} for k,v in STABILITY_RESULTS.items()},'validation_rows_read':0,'skipped_rows':0,'stage13_started':False},indent=2))

/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


{
  "artifact_root": "/content/drive/MyDrive/prelim/data/processed/mission2_raw_waveform_pretrained_speech/stage12_wavlm_canonical_reassessment_20261008T043852Z_fac003a8",
  "fresh_rows": 192,
  "prior_rows_excluded": 3015,
  "thresholds": {
    "cosine_minimum": 0.999,
    "cosine_median": 0.9999,
    "relative_l2_median": 0.005,
    "relative_l2_p99": 0.01,
    "coherent_mean_shift_ratio_max": 0.001,
    "nonfinite_allowed": 0,
    "skipped_rows_allowed": 0
  },
  "candidate_stability": {
    "next_power_of_two_samples": {
      "status": "FAIL",
      "aggregate": {
        "cosine_min": 0.9999065342196405,
        "cosine_median": 0.999997924462232,
        "relative_l2_median": 0.0020665228627181416,
        "relative_l2_p99": 0.011334651286353544,
        "mean_abs_mean": 0.00018663903128941664,
        "mean_abs_p99": 0.0008221160462739505,
        "max_abs_max": 0.009906530380249023,
        "coherent_mean_shift_ratio_max": 0.0009091857686979428
      },
      "old_unpadded_com

In [36]:
# CP5b-3 — Freeze the Stage 12 stopping decision from the saved preflight evidence.
# This reads existing artifacts only; it performs no new encoder inference or metric fitting.
import json, sys
from datetime import datetime, timezone
from pathlib import Path
saved=json.loads((ENGINEERING_ROOT/'batch_stability_report.json').read_text(encoding='utf-8'))
thresholds=saved['candidate_results']
limits=STABILITY_THRESHOLDS
candidate_summary={}
for rule,result in thresholds.items():
    failures=[]
    for condition,metrics in result['conditions'].items():
        failed=[]
        if metrics['cosine_min']<limits['cosine_minimum']: failed.append('cosine_minimum')
        if metrics['cosine_median']<limits['cosine_median']: failed.append('cosine_median')
        if metrics['relative_l2_median']>limits['relative_l2_median']: failed.append('relative_l2_median')
        if metrics['relative_l2_p99']>limits['relative_l2_p99']: failed.append('relative_l2_p99')
        if metrics['coherent_mean_shift_ratio_max']>limits['coherent_mean_shift_ratio_max']: failed.append('coherent_mean_shift_ratio_max')
        if not metrics['finite']: failed.append('nonfinite')
        if failed: failures.append({'condition':condition,'rows':metrics['comparisons'],'failed_limits':failed,'relative_l2_p99':metrics['relative_l2_p99'],'coherent_mean_shift_ratio_max':metrics['coherent_mean_shift_ratio_max']})
    candidate_summary[rule]={'status':result['status'],'conditions':len(result['conditions']),'failed_condition_count':len(failures),'failed_conditions':failures,'aggregate':result['aggregate'],'tested_batch_plan':result['tested_batch_plan']}
terminal={'schema':'mission2_stage12_canonical_preflight_stop_v1','created_utc':datetime.now(timezone.utc).isoformat(),'stage12_status':'PREFLIGHT FAILED — FROZEN REPRESENTATION BATCH-INVARIANCE GATE','performance':'NOT EVALUATED','stop_reason':'Both frozen sample-padding candidates failed at least one numerical batch-vs-singleton acceptance threshold. No throughput re-benchmark or full embedding extraction was started.','candidate_research_and_checkpoint':{'model_repository':saved['model_repository'],'model_revision':saved['model_revision']},'data_identity_fingerprint':saved['data_identity_fingerprint'],'source_split':'internal_train_only','fresh_stability_rows':saved['sample_rows'],'prior_rows_excluded':saved['duration_bins'],'validation_rows_read':saved['validation_rows_read'],'official_validation_used':saved['official_validation_used'],'thresholds':limits,'candidates':candidate_summary,'comparison_semantics_limit':'Saved stability conditions compare each condition embedding to the rule-specific singleton baseline. The report does not compute direct paired deltas between the first and repeated identical batch or between original and permuted companion batches; therefore isolated repeat/order equivalence is not claimed. The failed batch-size-vs-singleton conditions are sufficient to stop before throughput.','prior_compute_gate_preserved':{'decision':old_gate['decision'],'projected_full_hours':old_gate['projected_full_hours']},'new_drive_vs_scratch_io_benchmark':'NOT RUN','new_fresh_5k_to_10k_throughput_benchmark':'NOT RUN','new_projected_runtime':'NOT ESTIMATED','full_train_or_validation_embeddings':'NOT STARTED','classifier_fit_or_metrics':'NOT STARTED','stage13_started':False,'environment':{'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,'cuda':torch.version.cuda,'python':sys.version.split()[0]}}
stage12.write_json_new(ENGINEERING_ROOT/'preflight_stop_report.json',terminal)
print(json.dumps({'artifact_root':str(ENGINEERING_ROOT),'decision':terminal['stage12_status'],'performance':terminal['performance'],'candidate_failures':{k:{'status':v['status'],'failed':f"{v['failed_condition_count']}/{v['conditions']}"} for k,v in candidate_summary.items()},'new_throughput_benchmark':'NOT RUN','old_45.37h_gate_preserved':terminal['prior_compute_gate_preserved'],'stage13_started':False},ensure_ascii=False,indent=2))


{
  "artifact_root": "/content/drive/MyDrive/prelim/data/processed/mission2_raw_waveform_pretrained_speech/stage12_wavlm_canonical_reassessment_20261008T043852Z_fac003a8",
  "decision": "PREFLIGHT FAILED — FROZEN REPRESENTATION BATCH-INVARIANCE GATE",
  "performance": "NOT EVALUATED",
  "candidate_failures": {
    "250ms_multiple_4000_samples": {
      "status": "FAIL",
      "failed": "18/38"
    },
    "next_power_of_two_samples": {
      "status": "FAIL",
      "failed": "15/36"
    }
  },
  "new_throughput_benchmark": "NOT RUN",
  "old_45.37h_gate_preserved": {
    "decision": "RESOURCE/TIME-BUDGET INFEASIBLE",
    "projected_full_hours": 45.369999432472035
  },
  "stage13_started": false
}


In [37]:
# CP5b-4 — Confirm the existing Drive project log before appending a Stage 12 supplement.
from pathlib import Path
colab_repo_root=Path(OUTPUT_PARENT).resolve().parents[2]
colab_log_path=colab_repo_root/'docs'/'mission2_experiment_log.md'
print({'repo_root':str(colab_repo_root),'log_path':str(colab_log_path),'exists':colab_log_path.is_file(),'size_bytes':colab_log_path.stat().st_size if colab_log_path.is_file() else None})
if colab_log_path.is_file():
    text=colab_log_path.read_text(encoding='utf-8')
    print('already_has_this_run_marker', 'stage12_wavlm_canonical_reassessment_20261008T043852Z_fac003a8' in text)
    print('tail',text[-1600:])


{'repo_root': '/content/drive/MyDrive/prelim', 'log_path': '/content/drive/MyDrive/prelim/docs/mission2_experiment_log.md', 'exists': True, 'size_bytes': 59541}
already_has_this_run_marker False
tail w 5,000–10,000-row throughput benchmark was started, and no new runtime projection or compute-gate decision is claimed. The earlier batch-64 same-row result remains supplementary and does not replace CP5. Full embeddings/classifier evaluation remain unstarted; Stage 13 not started.


### Stage 12 supplementary equal-frame batching equivalence check — 2026-10-08

**Purpose:** one last representation-preserving microbatch check after raw-waveform and variable hidden-frame padding failed. The frozen feature extractor ran per utterance; only the Transformer encoder was batched for equal valid-frame lengths, with no hidden-frame padding.

- **Sample:** 8 internal-training rows drawn from the earlier 15-row equivalence diagnostic (4 pairs with equal valid-frame counts 24, 49, 99, and 249); no Va

In [38]:
# CP5b-5 — Append the verified preflight outcome without rewriting earlier Stage 12 records.
import json
from pathlib import Path
saved=json.loads((ENGINEERING_ROOT/'batch_stability_report.json').read_text(encoding='utf-8'))
terminal_saved=json.loads((ENGINEERING_ROOT/'preflight_stop_report.json').read_text(encoding='utf-8'))
log_path=colab_log_path
marker='### Stage 12 supplementary canonical padding stability gate — 2026-10-08'
current=log_path.read_text(encoding='utf-8')
if marker in current: raise FileExistsError('This exact Stage 12 supplement is already present; no log write performed.')
quarter=saved['candidate_results']['250ms_multiple_4000_samples']
power2=saved['candidate_results']['next_power_of_two_samples']
qstop=terminal_saved['candidates']['250ms_multiple_4000_samples']
pstop=terminal_saved['candidates']['next_power_of_two_samples']
section=f"""

{marker}

**Purpose:** re-evaluate whether deterministic sample-length padding and larger homogeneous batches can support a fresh Stage 12 throughput measurement. This is an engineering feasibility check; no accuracy was used to choose padding.

- **Checkpoint / data:** frozen microsoft/wavlm-base-plus@{saved['model_revision']}, internal Training only, data identity {saved['data_identity_fingerprint']}, clean 59D reference fingerprint 49d65a35d1e98c280f98e45cbed864ef8bac36268ccffbe81aa0985923c90de3, seed 42. The existing fixed split remains 711,467 train rows / 23,360 calls and 177,891 validation rows / 5,840 calls, with call overlap 0. The sample contained 192 fresh Training utterances across duration bins (32, 32, 64, 32, 32); 3,015 prior benchmark/diagnostic rows were excluded. Validation rows read: 0; Official Validation: not used.
- **Implementation correction:** the first stability-cell attempt stopped before producing a result because it incorrectly assumed the selected cohort shared one power-of-two pad target. The corrected rerun groups each candidate's forward batches by that candidate's deterministic target; it saved a new versioned run and did not overwrite the earlier gate or diagnostic artifacts.
- **Frozen acceptance thresholds:** cosine minimum >= 0.999; cosine median >= 0.9999; relative-L2 median <= 0.005; relative-L2 p99 <= 0.01; coherent mean-shift ratio <= 0.001; finite values only; skipped rows 0.
- **Measured candidate results:** 250 ms multiple-of-4,000-sample padding failed in {qstop['failed_condition_count']}/{qstop['conditions']} conditions; aggregate relative-L2 median/p99 {quarter['aggregate']['relative_l2_median']:.6f}/{quarter['aggregate']['relative_l2_p99']:.6f}, coherent mean-shift ratio max {quarter['aggregate']['coherent_mean_shift_ratio_max']:.6f}. Next-power-of-two padding failed in {pstop['failed_condition_count']}/{pstop['conditions']} conditions; aggregate relative-L2 median/p99 {power2['aggregate']['relative_l2_median']:.6f}/{power2['aggregate']['relative_l2_p99']:.6f}, coherent mean-shift ratio max {power2['aggregate']['coherent_mean_shift_ratio_max']:.6f}. Both candidates therefore failed the predeclared batch-vs-singleton representation stability gate.
- **Interpretation limit:** the saved conditions compare each condition to the rule-specific singleton baseline. Direct paired deltas between an original batch and its repeated batch, or original and permuted companion groups, were not computed; isolated repeat/order invariance is not claimed. Batch-vs-singleton failure alone stopped the workflow.
- **Stop:** no Drive-vs-scratch I/O comparison, fresh 5k–10k throughput benchmark, new runtime projection, or full extraction was started. Performance remains NOT EVALUATED. The earlier 45.3700-hour compute gate remains unchanged as a historical measurement and is not a projection for these unbenchmarked padding candidates. Stage 13 was not started.
- **Artifacts:** {ENGINEERING_ROOT.as_posix()} — stability_config.json, stability_sample_identity.json, batch_stability_report.json, candidate_comparison.json, preflight_stop_report.json.
"""
with log_path.open('a',encoding='utf-8',newline='') as f: f.write(section)
updated=log_path.read_text(encoding='utf-8')
if updated.count(marker)!=1 or not updated.rstrip().endswith(section.rstrip()): raise RuntimeError('Drive experiment-log append readback did not match the written section.')
print({'append_status':'VERIFIED','log_path':str(log_path),'marker_count':updated.count(marker),'size_bytes':log_path.stat().st_size,'artifact_root':str(ENGINEERING_ROOT),'stage12_status':terminal_saved['stage12_status'],'stage13_started':False})


{'append_status': 'VERIFIED', 'log_path': '/content/drive/MyDrive/prelim/docs/mission2_experiment_log.md', 'marker_count': 1, 'size_bytes': 62545, 'artifact_root': '/content/drive/MyDrive/prelim/data/processed/mission2_raw_waveform_pretrained_speech/stage12_wavlm_canonical_reassessment_20261008T043852Z_fac003a8', 'stage12_status': 'PREFLIGHT FAILED — FROZEN REPRESENTATION BATCH-INVARIANCE GATE', 'stage13_started': False}


In [39]:
# CP5b-6 — Correct one closure-report field without changing the measured stability results.
import json
v1_path=ENGINEERING_ROOT/'preflight_stop_report.json'
v1=json.loads(v1_path.read_text(encoding='utf-8'))
config=json.loads((ENGINEERING_ROOT/'stability_config.json').read_text(encoding='utf-8'))
stability=json.loads((ENGINEERING_ROOT/'batch_stability_report.json').read_text(encoding='utf-8'))
v2=dict(v1)
v2['schema']='mission2_stage12_canonical_preflight_stop_v2'
v2['supersedes']='preflight_stop_report.json; preserved unchanged because its prior_rows_excluded field was populated with duration-bin metadata by mistake.'
v2['prior_rows_excluded']=int(config['sample_plan']['excluded_prior_benchmark_or_diagnostic_rows'])
v2['sample_duration_bin_rows']={name:int(value['rows']) for name,value in stability['duration_bins'].items()}
v2['sample_duration_bins']=stability['duration_bins']
v2_path=ENGINEERING_ROOT/'preflight_stop_report_v2.json'
stage12.write_json_new(v2_path,v2)
old_line=f"- **Artifacts:** {ENGINEERING_ROOT.as_posix()} — stability_config.json, stability_sample_identity.json, batch_stability_report.json, candidate_comparison.json, preflight_stop_report.json."
new_line=f"- **Artifacts:** {ENGINEERING_ROOT.as_posix()} — stability_config.json, stability_sample_identity.json, batch_stability_report.json, candidate_comparison.json, preflight_stop_report_v2.json (canonical closure). The original preflight_stop_report.json remains preserved; v2 corrects its mislabeled prior-row-count field."
log_text=colab_log_path.read_text(encoding='utf-8')
if log_text.count(old_line)!=1: raise ValueError('Expected exactly one artifact line in the new Stage 12 section; no log replacement made.')
log_text=log_text.replace(old_line,new_line)
colab_log_path.write_text(log_text,encoding='utf-8')
readback=json.loads(v2_path.read_text(encoding='utf-8'))
verify_log=colab_log_path.read_text(encoding='utf-8')
if readback['prior_rows_excluded']!=3015 or readback['sample_duration_bin_rows']!={'<0.5':32,'0.5-1':32,'1-2':64,'2-5':32,'>=5':32} or new_line not in verify_log: raise RuntimeError('Correction artifact or Drive log readback failed.')
print({'status':'VERIFIED','canonical_stop_artifact':str(v2_path),'prior_rows_excluded':readback['prior_rows_excluded'],'sample_duration_bin_rows':readback['sample_duration_bin_rows'],'log_artifact_line_corrected':True,'encoder_inference_repeated':False})


{'status': 'VERIFIED', 'canonical_stop_artifact': '/content/drive/MyDrive/prelim/data/processed/mission2_raw_waveform_pretrained_speech/stage12_wavlm_canonical_reassessment_20261008T043852Z_fac003a8/preflight_stop_report_v2.json', 'prior_rows_excluded': 3015, 'sample_duration_bin_rows': {'0.5-1': 32, '1-2': 64, '2-5': 32, '<0.5': 32, '>=5': 32}, 'log_artifact_line_corrected': True, 'encoder_inference_repeated': False}


In [42]:
# CP5b-7 — Verify small synthetic contracts; pytest source is absent from this Drive repository copy.
import importlib.util, json
import numpy as np
from pathlib import Path
repo_root=Path(OUTPUT_PARENT).resolve().parents[2]
test_path=repo_root/'tests'/'test_mission2_raw_waveform_pretrained_speech.py'
assert not test_path.exists(), f'Unexpected test source appeared: {test_path}'
# Exercise the existing helper on synthetic arrays only; no WAV, model or real dataset is loaded.
padded,valid_lengths,padded_rows=stage12.pad_for_encoder([np.array([1.0,2.0],dtype=np.float32),np.array([3.0],dtype=np.float32)],sample_rate_hz=16000)
assert padded.shape==(2,400) and valid_lengths.tolist()==[2,1] and padded_rows==2
np.testing.assert_array_equal(padded[:,:2],[[1.0,2.0],[3.0,0.0]])
hidden=np.array([[[1.0],[3.0],[100.0]],[[4.0],[100.0],[100.0]]],dtype=np.float32)
frame_mask=np.array([[True,True,False],[True,False,False]])
np.testing.assert_array_equal(stage12.masked_mean_pool(hidden,frame_mask),[[2.0],[4.0]])
empty_waveform_rejected=False
try:
    stage12.pad_for_encoder([np.array([],dtype=np.float32)],sample_rate_hz=16000)
except ValueError:
    empty_waveform_rejected=True
assert empty_waveform_rejected
unit_test_report_v2={'schema':'mission2_stage12_unit_test_report_v2','status':'SYNTHETIC_CHECKS_PASS_PYTEST_NOT_RUN','pytest_status':'NOT_RUN','pytest_reason':'tests/test_mission2_raw_waveform_pretrained_speech.py is absent from the Colab Drive repository copy; pytest invocation found no tests and no dependency was installed.','supersedes':'unit_test_report.json records the failed discovery invocation (exit 4, no tests ran), not a test assertion failure.','synthetic_checks':{'fixed_minimum_padding_and_valid_lengths':'PASS','padding_values':'PASS','valid_frame_masked_mean':'PASS','zero_length_rejected':'PASS'},'real_data_loaded':False,'official_validation_used':False,'encoder_inference_performed':False,'full_extraction_started':False}
stage12.write_json_new(ENGINEERING_ROOT/'unit_test_report_v2.json',unit_test_report_v2)
print(json.dumps({'artifact':str(ENGINEERING_ROOT/'unit_test_report_v2.json'),**unit_test_report_v2},ensure_ascii=False,indent=2))


{
  "artifact": "/content/drive/MyDrive/prelim/data/processed/mission2_raw_waveform_pretrained_speech/stage12_wavlm_canonical_reassessment_20261008T043852Z_fac003a8/unit_test_report_v2.json",
  "schema": "mission2_stage12_unit_test_report_v2",
  "status": "SYNTHETIC_CHECKS_PASS_PYTEST_NOT_RUN",
  "pytest_status": "NOT_RUN",
  "pytest_reason": "tests/test_mission2_raw_waveform_pretrained_speech.py is absent from the Colab Drive repository copy; pytest invocation found no tests and no dependency was installed.",
  "supersedes": "unit_test_report.json records the failed discovery invocation (exit 4, no tests ran), not a test assertion failure.",
  "synthetic_checks": {
    "fixed_minimum_padding_and_valid_lengths": "PASS",
    "padding_values": "PASS",
    "valid_frame_masked_mean": "PASS",
    "zero_length_rejected": "PASS"
  },
  "real_data_loaded": false,
  "official_validation_used": false,
  "encoder_inference_performed": false,
  "full_extraction_started": false
}


In [43]:
# CP5b-8 — Record the synthetic-check result and pytest-source limitation in the Drive experiment log.
report=json.loads((ENGINEERING_ROOT/'unit_test_report_v2.json').read_text(encoding='utf-8'))
assert report['status']=='SYNTHETIC_CHECKS_PASS_PYTEST_NOT_RUN'
log_text=colab_log_path.read_text(encoding='utf-8')
artifact_lines=[line for line in log_text.splitlines() if line.startswith('- **Artifacts:**') and 'preflight_stop_report_v2.json' in line]
if len(artifact_lines)!=1: raise ValueError('Expected exactly one corrected Stage 12 artifact line.')
old_line=artifact_lines[0]
new_line=old_line.replace('preflight_stop_report_v2.json (canonical closure).','preflight_stop_report_v2.json (canonical closure), unit_test_report_v2.json.')
note='- **Offline synthetic helper checks:** PASS for minimum right-padding/valid lengths, padding values, valid-frame masked mean and zero-length rejection. The pytest suite was NOT RUN because its test file is absent from the Colab Drive copy; pytest reported no tests, exit 4, and no dependency was installed.'
updated=log_text.replace(old_line,new_line+'\n'+note)
colab_log_path.write_text(updated,encoding='utf-8')
verify=colab_log_path.read_text(encoding='utf-8')
if new_line not in verify or note not in verify or verify.count('Stage 12 supplementary canonical padding stability gate')!=1: raise RuntimeError('Drive log readback failed after unit-test note update.')
print({'status':'VERIFIED','log_path':str(colab_log_path),'synthetic_checks':report['synthetic_checks'],'pytest_status':'NOT_RUN','note_count':verify.count(note)})


{'status': 'VERIFIED', 'log_path': '/content/drive/MyDrive/prelim/docs/mission2_experiment_log.md', 'synthetic_checks': {'fixed_minimum_padding_and_valid_lengths': 'PASS', 'padding_values': 'PASS', 'valid_frame_masked_mean': 'PASS', 'zero_length_rejected': 'PASS'}, 'pytest_status': 'NOT_RUN', 'note_count': 1}


## CP6 — Full internal-train embedding extraction (gate PASS only)

Extract Training rows in source order to float32 shards. Each shard records row positions, keys, calls, labels, model revision, config/data fingerprints and checksum. Existing shards are resumed only after exact row identity/checksum verification; a completed shard written just before a disconnect can be recovered into the manifest. No validation row enters this stage.


In [ ]:
stage12.require_gate_pass(COMPUTE_GATE)
TRAIN_REPORT_PATH = RUN_ROOT / 'train_embedding_metadata.json'
if TRAIN_REPORT_PATH.exists():
    TRAIN_EXTRACTION = stage12.read_json(TRAIN_REPORT_PATH)
    if (TRAIN_EXTRACTION.get('status') != 'PASS'
            or TRAIN_EXTRACTION.get('expected_rows') != stage12.EXPECTED_SPLIT['train_rows']
            or TRAIN_EXTRACTION.get('config_hash') != CONFIG_HASH
            or TRAIN_EXTRACTION.get('data_identity_fingerprint') != context.identity['fingerprint']):
        raise ValueError('Saved train extraction report differs from the frozen run contract.')
else:
    TRAIN_EXTRACTION = stage12.extract_split(context, RUN_ROOT, 'train', model,
        config_hash=CONFIG_HASH, device='cuda', use_amp=True, batch_size=8, shard_rows=256)
    stage12.write_json_new(TRAIN_REPORT_PATH, TRAIN_EXTRACTION)
print(TRAIN_EXTRACTION)


## CP7 — Full fixed internal-validation embedding extraction

Only after complete CP6 train shards pass, extract the exact frozen internal Validation rows. This cell performs no loss, backpropagation, scaler fitting, checkpoint selection or gradient update. Any row/key/call/label/count/mask/dimension/finite-value mismatch stops.


In [ ]:
stage12.require_gate_pass(COMPUTE_GATE)
if TRAIN_EXTRACTION.get('status') != 'PASS' or TRAIN_EXTRACTION.get('expected_rows') != stage12.EXPECTED_SPLIT['train_rows']:
    raise RuntimeError('CP6 train extraction must pass before validation extraction.')
VAL_REPORT_PATH = RUN_ROOT / 'validation_embedding_metadata.json'
if VAL_REPORT_PATH.exists():
    VALIDATION_EXTRACTION = stage12.read_json(VAL_REPORT_PATH)
    if (VALIDATION_EXTRACTION.get('status') != 'PASS'
            or VALIDATION_EXTRACTION.get('expected_rows') != stage12.EXPECTED_SPLIT['validation_rows']
            or VALIDATION_EXTRACTION.get('config_hash') != CONFIG_HASH
            or VALIDATION_EXTRACTION.get('data_identity_fingerprint') != context.identity['fingerprint']):
        raise ValueError('Saved validation extraction report differs from the frozen run contract.')
else:
    VALIDATION_EXTRACTION = stage12.extract_split(context, RUN_ROOT, 'validation', model,
        config_hash=CONFIG_HASH, device='cuda', use_amp=True, batch_size=8, shard_rows=256)
    stage12.write_json_new(VAL_REPORT_PATH, VALIDATION_EXTRACTION)
print(VALIDATION_EXTRACTION)


## CP8 — Fixed linear probe and metrics

Fit StandardScaler incrementally on train embeddings only, then train the one fixed SGD logistic probe for five epochs with seed 42. Validation is transformed and evaluated only. Save train and validation probabilities/predictions, Accuracy, train–validation gap, confusion matrices, class recalls, FP/FN, errors and high-confidence errors (confidence ≥0.90). No tuning or threshold search.


In [ ]:
stage12.require_gate_pass(COMPUTE_GATE)
import joblib
import pandas as pd
CLASSIFIER_PATH = RUN_ROOT / 'classifier.joblib'
if CLASSIFIER_PATH.exists():
    saved_probe = joblib.load(CLASSIFIER_PATH)
    if (saved_probe.get('config_hash') != CONFIG_HASH
            or saved_probe.get('data_identity_fingerprint') != context.identity['fingerprint']
            or saved_probe.get('model_revision') != stage12.CONFIG['encoder']['revision']):
        raise ValueError('Saved classifier does not match this exact run.')
    scaler, classifier = saved_probe['scaler'], saved_probe['classifier']
    CLASSIFIER_REPORT = {'status':'PASS','artifact':str(CLASSIFIER_PATH),
        'artifact_sha256':stage12.file_sha256(CLASSIFIER_PATH),'scaler_fit_split':'internal_train_only',
        'classifier_fit_split':'internal_train_only','validation_fit_rows':0,'reused_exact_artifact':True}
else:
    scaler, classifier, CLASSIFIER_REPORT = stage12.fit_probe(RUN_ROOT, context, config_hash=CONFIG_HASH)
METRICS_PATH = RUN_ROOT / 'metrics.json'
if METRICS_PATH.exists():
    METRICS = stage12.read_json(METRICS_PATH)
    if (METRICS.get('config_hash') != CONFIG_HASH
            or METRICS.get('data_identity_fingerprint') != context.identity['fingerprint']):
        raise ValueError('Saved metrics belong to a different Stage 12 run.')
    PREDICTIONS = {}
    for split in ('train', 'validation'):
        frame = pd.read_parquet(RUN_ROOT / f'{split}_predictions.parquet')
        PREDICTIONS[split] = {column: frame[column].tolist() for column in
            ('record_key','call_id','y_true','probability','prediction','confidence','correct')}
else:
    METRICS, PREDICTIONS = stage12.evaluate_and_save(RUN_ROOT, context, scaler, classifier, config_hash=CONFIG_HASH)
print({'classifier':CLASSIFIER_REPORT,'metrics':METRICS})


## CP9 — Paired comparison with frozen TabM and Tabular ResNet

Read existing Stage 6 predictions only. Verify clean 59D fingerprint, seed/threshold/split, exact validation row/key/call/label alignment and canonical reference accuracy before paired counts. If reference artifacts are absent or differ, stop; do not refit or use historical duration/overlap slices.


In [ ]:
stage12.require_gate_pass(COMPUTE_GATE)
PAIRED_PATH = RUN_ROOT / 'paired_comparison.json'
if PAIRED_PATH.exists():
    PAIRED = stage12.read_json(PAIRED_PATH)
    if (PAIRED.get('rows') != stage12.EXPECTED_SPLIT['validation_rows']
            or PAIRED.get('config_hash') != CONFIG_HASH
            or PAIRED.get('data_identity_fingerprint') != context.identity['fingerprint']):
        raise ValueError('Saved paired comparison identity differs from this run.')
else:
    PAIRED = stage12.load_reference_predictions(REPO_ROOT, context, PREDICTIONS['validation'])
    PAIRED.update(config_hash=CONFIG_HASH, data_identity_fingerprint=context.identity['fingerprint'])
    stage12.write_json_new(PAIRED_PATH, PAIRED)
print(json.dumps(PAIRED, indent=2, ensure_ascii=False))


## CP10 — Reload, reproducibility, verified summary and stop

Reload the classifier artifact and regenerate train/validation probabilities from persisted float32 shards. Require exact keys/calls/labels and predictions, probabilities within `1e-12`, and unchanged Accuracy. Save `reload_report.json` and `summary.json`; only after all checks append Stage 12 to the experiment log. Stop here; do not begin Stage 13.


In [ ]:
stage12.require_gate_pass(COMPUTE_GATE)
RELOAD_PATH = RUN_ROOT / 'reload_report.json'
if RELOAD_PATH.exists():
    RELOAD = stage12.read_json(RELOAD_PATH)
    if (RELOAD.get('status') != 'PASS' or RELOAD.get('config_hash') != CONFIG_HASH
            or RELOAD.get('data_identity_fingerprint') != context.identity['fingerprint']
            or RELOAD.get('classifier_sha256') != stage12.file_sha256(RUN_ROOT / 'classifier.joblib')):
        raise ValueError('Existing reload report identity does not pass.')
else:
    RELOAD = stage12.reload_verify(RUN_ROOT, context, config_hash=CONFIG_HASH, original_predictions=PREDICTIONS)
SUMMARY = {
    'final_status':'PASS/FULL EXPERIMENT COMPLETED',
    'run_root':str(RUN_ROOT), 'config_hash':CONFIG_HASH,
    'encoder':CHECKPOINT_INFO, 'data_preflight':DATA_PREFLIGHT,
    'compute_gate':COMPUTE_GATE, 'throughput':THROUGHPUT,
    'train_extraction':TRAIN_EXTRACTION, 'validation_extraction':VALIDATION_EXTRACTION,
    'classifier':CLASSIFIER_REPORT, 'metrics':METRICS, 'paired_comparison':PAIRED, 'reload':RELOAD,
    'decision_rule':'If internal-validation Accuracy exceeds both frozen references, retain as a Stage 14 candidate only; otherwise hold. This does not select a final model.',
    'stage13_started':False}
SUMMARY_PATH = RUN_ROOT / 'summary.json'
if SUMMARY_PATH.exists():
    existing = stage12.read_json(SUMMARY_PATH)
    if existing.get('config_hash') != CONFIG_HASH or existing.get('final_status') != SUMMARY['final_status']:
        raise ValueError('Existing Stage 12 summary belongs to a different result.')
    SUMMARY = existing
else:
    stage12.write_json_new(SUMMARY_PATH, SUMMARY)
LOG_STATUS = stage12.append_result_to_log(REPO_ROOT, RUN_ROOT,
                                          'PASS/FULL EXPERIMENT COMPLETED', SUMMARY)
val_accuracy = METRICS['validation']['accuracy']
reference_max = max(stage12.REFERENCE_ACCURACY.values())
judgment = 'PROMOTE AS STAGE 14 CANDIDATE ONLY' if val_accuracy > reference_max else 'HOLD; no observed accuracy advantage over both frozen references'
print({'final_status':SUMMARY['final_status'], 'judgment':judgment,
       'internal_train_accuracy':METRICS['train']['accuracy'],
       'internal_validation_accuracy':val_accuracy,
       'train_validation_gap':METRICS['train_minus_validation_accuracy_gap'],
       'recall_0':METRICS['validation']['recall_0'], 'recall_1':METRICS['validation']['recall_1'],
       'confusion_matrix':METRICS['validation']['confusion_matrix'],
       'errors':METRICS['validation']['errors'], 'HCE':METRICS['validation']['hce'],
       'paired':PAIRED,'reload':RELOAD,'experiment_log':LOG_STATUS,
       'run_root':str(RUN_ROOT),'stage13':'NOT STARTED'})
